In [1]:
#!wget https://asr.iitm.ac.in/SPRING_INX/models/tokens_list/SPRING_INX_Gujarati_tokens.txt

In [2]:
# Cell 2: Load as TorchScript model (correct method)

import torch

PT_PATH = "/kaggle/input/models/nishargnargund/gujarati-asr/pytorch/default/1/SPRING_INX_streaming_k2_Gujarati.pt"

# Load as TorchScript
model = torch.jit.load(PT_PATH, map_location="cpu")
model.eval()

print(f"✅ Model loaded successfully!")
print(f"Type: {type(model)}")
print(f"\nModel class name: {model.original_name if hasattr(model, 'original_name') else 'N/A'}")

# List all methods available on the model
print(f"\nAvailable methods:")
for method in model._c._method_names():
    print(f"  - {method}")

✅ Model loaded successfully!
Type: <class 'torch.jit._script.RecursiveScriptModule'>

Model class name: AsrModel

Available methods:


In [3]:
# Cell 3: Deep inspect internals since _method_names() returned empty

import torch

# Try alternate ways to get methods
print("=== Methods via _c ===")
try:
    print(list(model._c._method_names()))
except Exception as e:
    print(f"  Error: {e}")

print("\n=== Methods via dir() ===")
methods = [m for m in dir(model) if not m.startswith('_')]
for m in methods:
    print(f"  - {m}")

print("\n=== Named children (submodules) ===")
for name, mod in model.named_children():
    print(f"  - {name}: {type(mod).__name__} | original_name: {getattr(mod, 'original_name', 'N/A')}")

print("\n=== Graph (forward) ===")
try:
    print(model.graph)
except Exception as e:
    print(f"  Error: {e}")

print("\n=== Code (forward) ===")
try:
    print(model.code)
except Exception as e:
    print(f"  Error: {e}")

=== Methods via _c ===
[]

=== Methods via dir() ===
  - T_destination
  - add_module
  - apply
  - bfloat16
  - buffers
  - call_super_init
  - children
  - code
  - code_with_constants
  - compile
  - cpu
  - cuda
  - decoder
  - define
  - double
  - dump_patches
  - encoder
  - encoder_embed
  - eval
  - extra_repr
  - float
  - forward
  - forward_magic_method
  - get_buffer
  - get_debug_state
  - get_extra_state
  - get_parameter
  - get_submodule
  - graph
  - graph_for
  - half
  - inlined_graph
  - ipu
  - joiner
  - load_state_dict
  - modules
  - mtia
  - named_buffers
  - named_children
  - named_modules
  - named_parameters
  - original_name
  - parameters
  - register_backward_hook
  - register_buffer
  - register_forward_hook
  - register_forward_pre_hook
  - register_full_backward_hook
  - register_full_backward_pre_hook
  - register_load_state_dict_post_hook
  - register_load_state_dict_pre_hook
  - register_module
  - register_parameter
  - register_state_dict_post_h

In [4]:
# Cell 4: Inspect each submodule individually

submodules = {
    "encoder_embed": model.encoder_embed,
    "encoder": model.encoder,
    "decoder": model.decoder,
    "joiner": model.joiner,
    "simple_am_proj": model.simple_am_proj,
    "simple_lm_proj": model.simple_lm_proj,
}

for name, mod in submodules.items():
    print(f"\n{'='*60}")
    print(f"SUBMODULE: {name} ({mod.original_name})")
    print(f"{'='*60}")
    
    # Methods
    try:
        methods = list(mod._c._method_names())
        print(f"Methods: {methods}")
    except:
        methods = []
        print("Methods: could not retrieve")
    
    # Code for each method
    for method in methods:
        print(f"\n--- {method} ---")
        try:
            print(getattr(mod, method).code)
        except Exception as e:
            print(f"  Error getting code: {e}")
    
    # If no methods, try .code directly
    if not methods:
        try:
            print(f"Code:\n{mod.code}")
        except Exception as e:
            print(f"  .code error: {e}")


SUBMODULE: encoder_embed (Conv2dSubsampling)
Methods: ['forward', 'get_init_states', 'streaming_forward']

--- forward ---
def forward(self,
    x: Tensor,
    x_lens: Tensor) -> Tuple[Tensor, Tensor]:
  x0 = torch.unsqueeze(x, 1)
  conv = self.conv
  x1 = (conv).forward(x0, )
  convnext = self.convnext
  x2 = (convnext).forward(x1, )
  b, c, t, f, = torch.size(x2)
  x3 = torch.reshape(torch.transpose(x2, 1, 2), [b, t, torch.mul(c, f)])
  out = self.out
  x4 = (out).forward(x3, )
  out_whiten = self.out_whiten
  x5 = (out_whiten).forward(x4, )
  out_norm = self.out_norm
  x6 = (out_norm).forward(x5, )
  dropout = self.dropout
  x7 = (dropout).forward(x6, )
  x_lens0 = torch.floor_divide(torch.sub(x_lens, 7), 2)
  _0 = torch.eq(torch.size(x7, 1), torch.item(torch.max(x_lens0)))
  if _0:
    pass
  else:
    _1 = (torch.size(x7, 1), torch.max(x_lens0))
    _2 = torch.add("AssertionError: ", str(_1))
    ops.prim.RaiseException(_2)
  return (x7, x_lens0)


--- get_init_states ---
def get

In [5]:
# Cell 5: Extract all shapes from parameters, buffers & key attributes

import torch

print("=== MODEL-LEVEL ATTRIBUTES ===")
for attr in ["chunk_size", "left_context_len"]:
    try:
        val = getattr(model.encoder, attr)
        print(f"  encoder.{attr} = {val}")
    except Exception as e:
        print(f"  encoder.{attr} -> Error: {e}")

print("\n=== ENCODER_EMBED PARAMETERS ===")
for name, param in model.encoder_embed.named_parameters():
    print(f"  {name}: {param.shape} | dtype: {param.dtype}")

print("\n=== ENCODER PARAMETERS (top-level only) ===")
for name, param in model.encoder.named_parameters():
    if name.count('.') <= 1:  # Only top-level to avoid flood
        print(f"  {name}: {param.shape} | dtype: {param.dtype}")

print("\n=== DECODER PARAMETERS ===")
for name, param in model.decoder.named_parameters():
    print(f"  {name}: {param.shape} | dtype: {param.dtype}")

print("\n=== JOINER PARAMETERS ===")
for name, param in model.joiner.named_parameters():
    print(f"  {name}: {param.shape} | dtype: {param.dtype}")

print("\n=== SIMPLE_AM_PROJ PARAMETERS ===")
for name, param in model.simple_am_proj.named_parameters():
    print(f"  {name}: {param.shape} | dtype: {param.dtype}")

print("\n=== SIMPLE_LM_PROJ PARAMETERS ===")
for name, param in model.simple_lm_proj.named_parameters():
    print(f"  {name}: {param.shape} | dtype: {param.dtype}")

print("\n=== DECODER ATTRIBUTES ===")
for attr in ["context_size"]:
    try:
        print(f"  decoder.{attr} = {model.decoder.context_size}")
    except Exception as e:
        print(f"  decoder.{attr} -> Error: {e}")

print("\n=== ENCODER_EMBED ATTRIBUTES ===")
for attr in ["out_width", "layer3_channels"]:
    try:
        val = getattr(model.encoder_embed, attr)
        print(f"  encoder_embed.{attr} = {val}")
    except Exception as e:
        print(f"  encoder_embed.{attr} -> Error: {e}")

=== MODEL-LEVEL ATTRIBUTES ===
  encoder.chunk_size = 32
  encoder.left_context_len = 128

=== ENCODER_EMBED PARAMETERS ===
  conv.0.weight: torch.Size([8, 1, 3, 3]) | dtype: torch.float32
  conv.0.bias: torch.Size([8]) | dtype: torch.float32
  conv.4.weight: torch.Size([32, 8, 3, 3]) | dtype: torch.float32
  conv.4.bias: torch.Size([32]) | dtype: torch.float32
  conv.7.weight: torch.Size([128, 32, 3, 3]) | dtype: torch.float32
  conv.7.bias: torch.Size([128]) | dtype: torch.float32
  convnext.depthwise_conv.weight: torch.Size([128, 1, 7, 7]) | dtype: torch.float32
  convnext.depthwise_conv.bias: torch.Size([128]) | dtype: torch.float32
  convnext.pointwise_conv1.weight: torch.Size([384, 128, 1, 1]) | dtype: torch.float32
  convnext.pointwise_conv1.bias: torch.Size([384]) | dtype: torch.float32
  convnext.pointwise_conv2.weight: torch.Size([128, 384, 1, 1]) | dtype: torch.float32
  convnext.pointwise_conv2.bias: torch.Size([128]) | dtype: torch.float32
  out.weight: torch.Size([192, 24

In [6]:
# Cell 6: Get init states and inspect shapes

import torch

BATCH_SIZE = 1
device = torch.device("cpu")

# Get initial states from encoder
states = model.encoder.get_init_states(batch_size=BATCH_SIZE, device=device)

print(f"Number of states: {len(states)}")
print(f"\nState shapes:")
for i, s in enumerate(states):
    print(f"  states[{i}]: shape={s.shape} | dtype={s.dtype}")

# Also check encoder_embed init state separately
embed_state = model.encoder_embed.get_init_states(batch_size=BATCH_SIZE, device=device)
print(f"\nencoder_embed init state: shape={embed_state.shape} | dtype={embed_state.dtype}")

# Key derived values summary
print("\n=== ARCHITECTURE SUMMARY ===")
print(f"  chunk_size         : {model.encoder.chunk_size}")
print(f"  left_context_len   : {model.encoder.left_context_len}")
print(f"  vocab_size         : 900  (from embedding & joiner output)")
print(f"  encoder_dim        : 512  (from joiner encoder_proj input)")
print(f"  decoder_dim        : 512  (from joiner decoder_proj input)")
print(f"  context_size       : {model.decoder.context_size}")
print(f"  encoder_embed out  : 192  (from out.bias)")
print(f"  out_width          : {model.encoder_embed.out_width}")
print(f"  layer3_channels    : {model.encoder_embed.layer3_channels}")

Number of states: 98

State shapes:
  states[0]: shape=torch.Size([128, 1, 128]) | dtype=torch.float32
  states[1]: shape=torch.Size([1, 1, 128, 144]) | dtype=torch.float32
  states[2]: shape=torch.Size([128, 1, 48]) | dtype=torch.float32
  states[3]: shape=torch.Size([128, 1, 48]) | dtype=torch.float32
  states[4]: shape=torch.Size([1, 192, 15]) | dtype=torch.float32
  states[5]: shape=torch.Size([1, 192, 15]) | dtype=torch.float32
  states[6]: shape=torch.Size([128, 1, 128]) | dtype=torch.float32
  states[7]: shape=torch.Size([1, 1, 128, 144]) | dtype=torch.float32
  states[8]: shape=torch.Size([128, 1, 48]) | dtype=torch.float32
  states[9]: shape=torch.Size([128, 1, 48]) | dtype=torch.float32
  states[10]: shape=torch.Size([1, 192, 15]) | dtype=torch.float32
  states[11]: shape=torch.Size([1, 192, 15]) | dtype=torch.float32
  states[12]: shape=torch.Size([64, 1, 128]) | dtype=torch.float32
  states[13]: shape=torch.Size([1, 1, 64, 192]) | dtype=torch.float32
  states[14]: shape=tor

In [7]:
# Cell 7: Full dummy forward pass through each submodule

import torch

BATCH_SIZE = 1
device = torch.device("cpu")

# === KEY DIMENSIONS (from our scraping) ===
# chunk_size = 32  → encoder expects exactly 32 frames after subsampling
# subsampling formula: x_lens_out = (x_lens - 7) // 2 - 3  (streaming)
# So input frames needed: chunk_size * 2 + 7 + 6 = 32*2 + 13 = 77 raw frames
# Feature dim = 80 (standard mel filterbank — we'll verify via conv input)
# vocab_size = 900, context_size = 2

CHUNK_SIZE = 32
FEAT_DIM = 80          # Standard mel — conv expects [B, T, F]
# Raw frames: chunk_size*2 + 7 + 3*2 = 77
RAW_FRAMES = 77        # (77-7)//2=35, 35-3=32 ✓ matches chunk_size

print("=== STEP 1: Get init states ===")
states = model.encoder.get_init_states(batch_size=BATCH_SIZE, device=device)
print(f"  states count: {len(states)}")
print(f"  states[-2] (embed cache): {states[-2].shape}")
print(f"  states[-1] (processed_lens): {states[-1].shape} | dtype: {states[-1].dtype}")

print("\n=== STEP 2: Dummy input ===")
features = torch.randn(BATCH_SIZE, RAW_FRAMES, FEAT_DIM)
feature_lengths = torch.tensor([RAW_FRAMES] * BATCH_SIZE, dtype=torch.int32)
print(f"  features:        {features.shape}")
print(f"  feature_lengths: {feature_lengths}")

print("\n=== STEP 3: encoder_embed.streaming_forward ===")
cached_left_pad = states[-2]
x, x_lens, new_cached_left_pad = model.encoder_embed.streaming_forward(
    features, feature_lengths, cached_left_pad
)
print(f"  x:                   {x.shape}")
print(f"  x_lens:              {x_lens}")
print(f"  new_cached_left_pad: {new_cached_left_pad.shape}")

print("\n=== STEP 4: encoder.forward (full streaming) ===")
encoder_out, encoder_out_lens, new_states = model.encoder.forward(
    features, feature_lengths, states
)
print(f"  encoder_out:      {encoder_out.shape}")
print(f"  encoder_out_lens: {encoder_out_lens}")
print(f"  new_states count: {len(new_states)}")

print("\n=== STEP 5: decoder.forward ===")
# context_size=2, vocab_size=900
# y shape: [B, context_size] with token ids (0 = blank)
y = torch.zeros(BATCH_SIZE, 2, dtype=torch.int64)
decoder_out = model.decoder.forward(y, need_pad=False)
print(f"  y:           {y.shape}")
print(f"  decoder_out: {decoder_out.shape}")

print("\n=== STEP 6: joiner.forward ===")
# encoder_out: [B, T, 512] → need [B, T, 1, 512]
# decoder_out: [B, context_size, 512] → need [B, 1, U, 512]
enc = encoder_out.unsqueeze(2)   # [B, T, 1, 512]
dec = decoder_out.unsqueeze(1)   # [B, 1, U, 512]
joiner_out = model.joiner.forward(enc, dec, project_input=True)
print(f"  encoder input to joiner: {enc.shape}")
print(f"  decoder input to joiner: {dec.shape}")
print(f"  joiner_out:              {joiner_out.shape}")

print("\n=== STEP 7: simple_am_proj & simple_lm_proj ===")
am_out = model.simple_am_proj.forward(encoder_out)
lm_out = model.simple_lm_proj.forward(decoder_out)
print(f"  am_out (simple_am_proj): {am_out.shape}")
print(f"  lm_out (simple_lm_proj): {lm_out.shape}")

print("\n✅ ALL FORWARD PASSES SUCCESSFUL")
print("\n=== FINAL SHAPE SUMMARY ===")
print(f"  Input features:    [{BATCH_SIZE}, {RAW_FRAMES}, {FEAT_DIM}]")
print(f"  Encoder output:    {encoder_out.shape}")
print(f"  Decoder output:    {decoder_out.shape}")
print(f"  Joiner output:     {joiner_out.shape}  ← logits over vocab")
print(f"  States in:         98 tensors")
print(f"  States out:        {len(new_states)} tensors")

=== STEP 1: Get init states ===
  states count: 98
  states[-2] (embed cache): torch.Size([1, 128, 3, 19])
  states[-1] (processed_lens): torch.Size([1]) | dtype: torch.int32

=== STEP 2: Dummy input ===
  features:        torch.Size([1, 77, 80])
  feature_lengths: tensor([77], dtype=torch.int32)

=== STEP 3: encoder_embed.streaming_forward ===
  x:                   torch.Size([1, 32, 192])
  x_lens:              tensor([32], dtype=torch.int32)
  new_cached_left_pad: torch.Size([1, 128, 3, 19])

=== STEP 4: encoder.forward (full streaming) ===
  encoder_out:      torch.Size([1, 16, 512])
  encoder_out_lens: tensor([16], dtype=torch.int32)
  new_states count: 98

=== STEP 5: decoder.forward ===
  y:           torch.Size([1, 2])
  decoder_out: torch.Size([1, 1, 512])

=== STEP 6: joiner.forward ===
  encoder input to joiner: torch.Size([1, 16, 1, 512])
  decoder input to joiner: torch.Size([1, 1, 1, 512])
  joiner_out:              torch.Size([1, 16, 1, 900])

=== STEP 7: simple_am_proj

In [8]:
# Cell 8: Print complete verified shape manifest for ONNX export reference

print("=" * 65)
print("   SPRING-INX GUJARATI ASR — VERIFIED SHAPE MANIFEST")
print("=" * 65)

print("""
┌─────────────────────────────────────────────────────────────┐
│  ARCHITECTURE: RNN-T (Transducer) + Streaming Zipformer     │
│  SUBMODULES: encoder_embed → encoder → decoder → joiner     │
└─────────────────────────────────────────────────────────────┘

── GLOBAL CONSTANTS ──────────────────────────────────────────
  vocab_size        = 900
  feat_dim          = 80   (mel filterbanks)
  chunk_size        = 32   (raw feature frames after embed)
  left_context_len  = 128
  context_size      = 2    (decoder history tokens)
  encoder_dim       = 512
  encoder_embed_dim = 192

── INPUTS ────────────────────────────────────────────────────
  features          : [B, 77, 80]   float32
                       └─ 77 = chunk_size*2 + 13 (streaming formula)
  feature_lengths   : [B]           int32

── STATES (98 tensors, List[Tensor]) ─────────────────────────
  states[0..95]     : encoder Zipformer layer caches
  states[96]        : embed cache  [B, 128, 3, 19]  float32
  states[97]        : processed_lens  [B]            int32

  State shape groups (6 tensors per Zipformer block):
    attn_cache      : [left_ctx, B, head_dim]
    attn_key_cache  : [B, 1, left_ctx, head_dim*3/2]
    conv_cache_1    : [left_ctx, B, conv_dim]
    conv_cache_2    : [left_ctx, B, conv_dim]
    bypa_cache_1    : [B, channels, kernel]
    bypa_cache_2    : [B, channels, kernel]

── ENCODER OUTPUT ────────────────────────────────────────────
  encoder_out       : [B, 16, 512]  float32
                       └─ 16 = chunk_size // 2 (internal downsampling)
  encoder_out_lens  : [B]           int32
  new_states        : 98 tensors    (same shapes as input states)

── DECODER ───────────────────────────────────────────────────
  INPUT  y          : [B, context_size=2]   int64
  OUTPUT decoder_out: [B, 1, 512]           float32

── JOINER ────────────────────────────────────────────────────
  INPUT  enc        : [B, T=16, 1,    512]  float32
  INPUT  dec        : [B, 1,    U=1,  512]  float32
  OUTPUT joiner_out : [B, T=16, U=1,  900]  float32

── PROJECTIONS ───────────────────────────────────────────────
  simple_am_proj    : [B, T, 512] → [B, T, 900]
  simple_lm_proj    : [B, U, 512] → [B, U, 900]
""")

print("── ONNX EXPORT STRATEGY ──────────────────────────────────────")
print("""
  We will export 3 SEPARATE ONNX models (standard k2/icefall pattern):

  1. encoder.onnx
     IN  : features [B,77,80], feature_lengths [B], states[0..97]
     OUT : encoder_out [B,16,512], encoder_out_lens [B], new_states[0..97]
     Dynamic axes: B (batch), states batch dim

  2. decoder.onnx
     IN  : y [B, 2]
     OUT : decoder_out [B, 1, 512]
     Dynamic axes: B (batch)

  3. joiner.onnx
     IN  : encoder_out [B,T,1,512], decoder_out [B,1,U,512]
     OUT : logits [B,T,U,900]
     Dynamic axes: B, T, U

  WHY 3 MODELS:
  - Beam search loop runs decoder+joiner per token step
  - Encoder runs once per chunk
  - Splitting avoids dynamic control flow issues in ONNX
""")

print("── POTENTIAL ONNX RISK POINTS ────────────────────────────────")
print("""
  ⚠  98 state tensors → will be flattened as individual ONNX inputs/outputs
  ⚠  make_pad_mask (custom icefall op) → need to verify it traces cleanly
  ⚠  states[-1] is int32 processed_lens → must preserve dtype in ONNX
  ⚠  encoder internally halves T: input chunk=32 → output T=16
  ⚠  need_pad=False for decoder in inference mode
  ⚠  project_input=True for joiner in inference mode
""")

print("Ready for Cell 9: Export encoder.onnx ✅")

   SPRING-INX GUJARATI ASR — VERIFIED SHAPE MANIFEST

┌─────────────────────────────────────────────────────────────┐
│  ARCHITECTURE: RNN-T (Transducer) + Streaming Zipformer     │
│  SUBMODULES: encoder_embed → encoder → decoder → joiner     │
└─────────────────────────────────────────────────────────────┘

── GLOBAL CONSTANTS ──────────────────────────────────────────
  vocab_size        = 900
  feat_dim          = 80   (mel filterbanks)
  chunk_size        = 32   (raw feature frames after embed)
  left_context_len  = 128
  context_size      = 2    (decoder history tokens)
  encoder_dim       = 512
  encoder_embed_dim = 192

── INPUTS ────────────────────────────────────────────────────
  features          : [B, 77, 80]   float32
                       └─ 77 = chunk_size*2 + 13 (streaming formula)
  feature_lengths   : [B]           int32

── STATES (98 tensors, List[Tensor]) ─────────────────────────
  states[0..95]     : encoder Zipformer layer caches
  states[96]        : embed c

In [9]:
!pip install onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 68.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.3 MB/s eta 0:00:00


In [10]:
# Cell 11: Use torch.jit.script wrapper to properly expose the encoder

import torch
import os

BATCH_SIZE = 1
RAW_FRAMES = 77
FEAT_DIM = 80

features = torch.randn(BATCH_SIZE, RAW_FRAMES, FEAT_DIM)
feature_lengths = torch.tensor([RAW_FRAMES] * BATCH_SIZE, dtype=torch.int32)
init_states = model.encoder.get_init_states(batch_size=BATCH_SIZE, device=torch.device("cpu"))

# Since model.encoder is already a TorchScript module, export it DIRECTLY
# using torch.onnx.export with the scripted module itself
# Pass states as a list (matching the original forward signature)

class EncoderONNXWrapper(torch.nn.Module):
    def __init__(self, encoder):
        super().__init__()
        # Register as submodule so tracer can see it
        self.encoder = encoder

    def forward(self,
                features: torch.Tensor,
                feature_lengths: torch.Tensor,
                state_in_0: torch.Tensor, state_in_1: torch.Tensor,
                state_in_2: torch.Tensor, state_in_3: torch.Tensor,
                state_in_4: torch.Tensor, state_in_5: torch.Tensor,
                state_in_6: torch.Tensor, state_in_7: torch.Tensor,
                state_in_8: torch.Tensor, state_in_9: torch.Tensor,
                state_in_10: torch.Tensor, state_in_11: torch.Tensor,
                state_in_12: torch.Tensor, state_in_13: torch.Tensor,
                state_in_14: torch.Tensor, state_in_15: torch.Tensor,
                state_in_16: torch.Tensor, state_in_17: torch.Tensor,
                state_in_18: torch.Tensor, state_in_19: torch.Tensor,
                state_in_20: torch.Tensor, state_in_21: torch.Tensor,
                state_in_22: torch.Tensor, state_in_23: torch.Tensor,
                state_in_24: torch.Tensor, state_in_25: torch.Tensor,
                state_in_26: torch.Tensor, state_in_27: torch.Tensor,
                state_in_28: torch.Tensor, state_in_29: torch.Tensor,
                state_in_30: torch.Tensor, state_in_31: torch.Tensor,
                state_in_32: torch.Tensor, state_in_33: torch.Tensor,
                state_in_34: torch.Tensor, state_in_35: torch.Tensor,
                state_in_36: torch.Tensor, state_in_37: torch.Tensor,
                state_in_38: torch.Tensor, state_in_39: torch.Tensor,
                state_in_40: torch.Tensor, state_in_41: torch.Tensor,
                state_in_42: torch.Tensor, state_in_43: torch.Tensor,
                state_in_44: torch.Tensor, state_in_45: torch.Tensor,
                state_in_46: torch.Tensor, state_in_47: torch.Tensor,
                state_in_48: torch.Tensor, state_in_49: torch.Tensor,
                state_in_50: torch.Tensor, state_in_51: torch.Tensor,
                state_in_52: torch.Tensor, state_in_53: torch.Tensor,
                state_in_54: torch.Tensor, state_in_55: torch.Tensor,
                state_in_56: torch.Tensor, state_in_57: torch.Tensor,
                state_in_58: torch.Tensor, state_in_59: torch.Tensor,
                state_in_60: torch.Tensor, state_in_61: torch.Tensor,
                state_in_62: torch.Tensor, state_in_63: torch.Tensor,
                state_in_64: torch.Tensor, state_in_65: torch.Tensor,
                state_in_66: torch.Tensor, state_in_67: torch.Tensor,
                state_in_68: torch.Tensor, state_in_69: torch.Tensor,
                state_in_70: torch.Tensor, state_in_71: torch.Tensor,
                state_in_72: torch.Tensor, state_in_73: torch.Tensor,
                state_in_74: torch.Tensor, state_in_75: torch.Tensor,
                state_in_76: torch.Tensor, state_in_77: torch.Tensor,
                state_in_78: torch.Tensor, state_in_79: torch.Tensor,
                state_in_80: torch.Tensor, state_in_81: torch.Tensor,
                state_in_82: torch.Tensor, state_in_83: torch.Tensor,
                state_in_84: torch.Tensor, state_in_85: torch.Tensor,
                state_in_86: torch.Tensor, state_in_87: torch.Tensor,
                state_in_88: torch.Tensor, state_in_89: torch.Tensor,
                state_in_90: torch.Tensor, state_in_91: torch.Tensor,
                state_in_92: torch.Tensor, state_in_93: torch.Tensor,
                state_in_94: torch.Tensor, state_in_95: torch.Tensor,
                state_in_96: torch.Tensor, state_in_97: torch.Tensor,
                ) -> torch.Tensor:

        states_list = [
            state_in_0, state_in_1, state_in_2, state_in_3,
            state_in_4, state_in_5, state_in_6, state_in_7,
            state_in_8, state_in_9, state_in_10, state_in_11,
            state_in_12, state_in_13, state_in_14, state_in_15,
            state_in_16, state_in_17, state_in_18, state_in_19,
            state_in_20, state_in_21, state_in_22, state_in_23,
            state_in_24, state_in_25, state_in_26, state_in_27,
            state_in_28, state_in_29, state_in_30, state_in_31,
            state_in_32, state_in_33, state_in_34, state_in_35,
            state_in_36, state_in_37, state_in_38, state_in_39,
            state_in_40, state_in_41, state_in_42, state_in_43,
            state_in_44, state_in_45, state_in_46, state_in_47,
            state_in_48, state_in_49, state_in_50, state_in_51,
            state_in_52, state_in_53, state_in_54, state_in_55,
            state_in_56, state_in_57, state_in_58, state_in_59,
            state_in_60, state_in_61, state_in_62, state_in_63,
            state_in_64, state_in_65, state_in_66, state_in_67,
            state_in_68, state_in_69, state_in_70, state_in_71,
            state_in_72, state_in_73, state_in_74, state_in_75,
            state_in_76, state_in_77, state_in_78, state_in_79,
            state_in_80, state_in_81, state_in_82, state_in_83,
            state_in_84, state_in_85, state_in_86, state_in_87,
            state_in_88, state_in_89, state_in_90, state_in_91,
            state_in_92, state_in_93, state_in_94, state_in_95,
            state_in_96, state_in_97,
        ]
        encoder_out, encoder_out_lens, new_states = self.encoder.forward(
            features, feature_lengths, states_list
        )
        # Stack all outputs into one tensor to sidestep tuple export issues
        # We'll return individually via a scripted module
        return encoder_out

# First test that submodule registration works
wrapper = EncoderONNXWrapper(model.encoder)
wrapper.eval()

# Script it so the tracer can see the submodule
scripted_wrapper = torch.jit.script(wrapper)

with torch.no_grad():
    test = scripted_wrapper(features, feature_lengths, *init_states)
print(f"Scripted wrapper test: {test.shape}")
print("✅ Scripting succeeded — ready for proper multi-output export in Cell 12")

Scripted wrapper test: torch.Size([1, 16, 512])
✅ Scripting succeeded — ready for proper multi-output export in Cell 12


In [11]:
# Cell 14: Avoid large tuple — use torch.stack to bundle states by shape group

import torch
import os
from typing import Tuple, List

# Key insight: states come in repeating shape groups.
# We'll pass them as-is but return encoder_out + stacked states blob
# by padding all states to same size, stacking, then unstacking on device.
# 
# BETTER approach: just use torch.jit.trace (not script) on the wrapper
# since the encoder is already a TorchScript module — trace doesn't need
# return type annotations.

class EncoderWrapperTrace(torch.nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder  # already a ScriptModule — registered as submodule

    def forward(self, features, feature_lengths, states):
        # states is a List[Tensor] — trace handles this fine
        encoder_out, encoder_out_lens, new_states = self.encoder.forward(
            features, feature_lengths, states
        )
        return encoder_out, encoder_out_lens, new_states

BATCH_SIZE  = 1
RAW_FRAMES  = 77
FEAT_DIM    = 80

features        = torch.randn(BATCH_SIZE, RAW_FRAMES, FEAT_DIM)
feature_lengths = torch.tensor([RAW_FRAMES] * BATCH_SIZE, dtype=torch.int32)
init_states     = model.encoder.get_init_states(batch_size=BATCH_SIZE, device=torch.device("cpu"))

wrapper = EncoderWrapperTrace(model.encoder)
wrapper.eval()

# ── Trace (not script) ──
print("Tracing wrapper ...")
with torch.no_grad():
    traced = torch.jit.trace(
        wrapper,
        (features, feature_lengths, init_states),
        strict=False,
    )

# ── Verify traced output ──
with torch.no_grad():
    enc_out, enc_lens, new_st = traced(features, feature_lengths, init_states)
print(f"encoder_out:      {enc_out.shape}")
print(f"encoder_out_lens: {enc_lens}")
print(f"new_states count: {len(new_st)}")
print(f"new_states[0]:    {new_st[0].shape}")
print(f"new_states[97]:   {new_st[97].shape}")

# ── Now export traced module ──
# For ONNX we need flat inputs — build a flat wrapper around the traced module
print("\nNow building flat ONNX-exportable wrapper around traced module ...")

# Save traced module first, reload as script
traced.save("encoder_traced.pt")
traced_loaded = torch.jit.load("encoder_traced.pt")
print("✅ Traced module saved and reloaded successfully")
print(f"   Methods: {list(traced_loaded._c._method_names())}")

Tracing wrapper ...
encoder_out:      torch.Size([1, 16, 512])
encoder_out_lens: tensor([16], dtype=torch.int32)
new_states count: 98
new_states[0]:    torch.Size([128, 1, 128])
new_states[97]:   torch.Size([1])

Now building flat ONNX-exportable wrapper around traced module ...
✅ Traced module saved and reloaded successfully
   Methods: ['forward']


In [12]:
# Cell 19: Bypass dynamic state indexing by splitting encoder into two parts
# The encoder does states[-2] and states[-1] internally — we pre-extract
# those and pass them separately, giving the inner Zipformer only states[0:96]

import torch
import torch.nn as nn
import os, time

BATCH_SIZE  = 1
RAW_FRAMES  = 77
FEAT_DIM    = 80

class PatchedEncoderWrapper(nn.Module):
    """
    Manually replicates StreamingEncoderModel.forward() but with
    ALL state indexing done in Python (not recorded by tracer).
    
    Original forward does:
        cached_embed_left_pad = states[-2]   ← dynamic index → ONNX crash
        processed_lens        = states[-1]   ← dynamic index → ONNX crash
        encoder_states        = states[:-2]  ← dynamic slice  → ONNX crash
    
    We pre-extract all three here in Python before any tracing begins.
    """
    def __init__(self, encoder_model):
        super().__init__()
        # Register submodules individually so tracer can see them
        self.encoder_embed = encoder_model.encoder_embed
        self.encoder       = encoder_model.encoder   # inner Zipformer

    def forward(self,
                features,           # [B, 77, 80]
                feature_lengths,    # [B]
                # Inner encoder states (96 tensors, states[0:96])
                s0,  s1,  s2,  s3,  s4,  s5,  s6,  s7,
                s8,  s9,  s10, s11, s12, s13, s14, s15,
                s16, s17, s18, s19, s20, s21, s22, s23,
                s24, s25, s26, s27, s28, s29, s30, s31,
                s32, s33, s34, s35, s36, s37, s38, s39,
                s40, s41, s42, s43, s44, s45, s46, s47,
                s48, s49, s50, s51, s52, s53, s54, s55,
                s56, s57, s58, s59, s60, s61, s62, s63,
                s64, s65, s66, s67, s68, s69, s70, s71,
                s72, s73, s74, s75, s76, s77, s78, s79,
                s80, s81, s82, s83, s84, s85, s86, s87,
                s88, s89, s90, s91, s92, s93, s94, s95,
                # state[96]: embed cache [B, 128, 3, 19]
                cached_embed_left_pad,
                # state[97]: processed_lens [B] int32
                processed_lens,
                ):

        chunk_size       = self.encoder.chunk_size
        left_context_len = self.encoder.left_context_len

        # ── encoder_embed streaming forward ──
        x, x_lens, new_cached_embed_left_pad = self.encoder_embed.streaming_forward(
            features, feature_lengths, cached_embed_left_pad
        )

        # ── padding mask (replicated from encoder forward) ──
        src_key_padding_mask = torch.zeros(
            x.shape[0], x_lens.shape[0], dtype=torch.bool
        )  # simplified: all valid for fixed chunk

        # processed_lens mask
        _arange = torch.arange(left_context_len, device=x.device).unsqueeze(0)
        _arange = _arange.expand(x.shape[0], left_context_len)
        processed_mask = torch.le(processed_lens.unsqueeze(1), _arange)
        processed_mask0 = torch.flip(processed_mask, [1])
        new_processed_lens = processed_lens + x_lens

        src_key_padding_mask0 = torch.cat([processed_mask0, src_key_padding_mask], dim=1)

        # ── inner Zipformer encoder ──
        # Assemble encoder_states list in Python (not traced)
        encoder_states = [
            s0,  s1,  s2,  s3,  s4,  s5,  s6,  s7,
            s8,  s9,  s10, s11, s12, s13, s14, s15,
            s16, s17, s18, s19, s20, s21, s22, s23,
            s24, s25, s26, s27, s28, s29, s30, s31,
            s32, s33, s34, s35, s36, s37, s38, s39,
            s40, s41, s42, s43, s44, s45, s46, s47,
            s48, s49, s50, s51, s52, s53, s54, s55,
            s56, s57, s58, s59, s60, s61, s62, s63,
            s64, s65, s66, s67, s68, s69, s70, s71,
            s72, s73, s74, s75, s76, s77, s78, s79,
            s80, s81, s82, s83, s84, s85, s86, s87,
            s88, s89, s90, s91, s92, s93, s94, s95,
        ]

        x0 = torch.permute(x, [1, 0, 2])
        encoder_out, encoder_out_lens, new_encoder_states = self.encoder.streaming_forward(
            x0, x_lens, encoder_states, src_key_padding_mask0
        )
        encoder_out0 = torch.permute(encoder_out, [1, 0, 2])

        return (encoder_out0, encoder_out_lens,
                new_encoder_states[0],  new_encoder_states[1],
                new_encoder_states[2],  new_encoder_states[3],
                new_encoder_states[4],  new_encoder_states[5],
                new_encoder_states[6],  new_encoder_states[7],
                new_encoder_states[8],  new_encoder_states[9],
                new_encoder_states[10], new_encoder_states[11],
                new_encoder_states[12], new_encoder_states[13],
                new_encoder_states[14], new_encoder_states[15],
                new_encoder_states[16], new_encoder_states[17],
                new_encoder_states[18], new_encoder_states[19],
                new_encoder_states[20], new_encoder_states[21],
                new_encoder_states[22], new_encoder_states[23],
                new_encoder_states[24], new_encoder_states[25],
                new_encoder_states[26], new_encoder_states[27],
                new_encoder_states[28], new_encoder_states[29],
                new_encoder_states[30], new_encoder_states[31],
                new_encoder_states[32], new_encoder_states[33],
                new_encoder_states[34], new_encoder_states[35],
                new_encoder_states[36], new_encoder_states[37],
                new_encoder_states[38], new_encoder_states[39],
                new_encoder_states[40], new_encoder_states[41],
                new_encoder_states[42], new_encoder_states[43],
                new_encoder_states[44], new_encoder_states[45],
                new_encoder_states[46], new_encoder_states[47],
                new_encoder_states[48], new_encoder_states[49],
                new_encoder_states[50], new_encoder_states[51],
                new_encoder_states[52], new_encoder_states[53],
                new_encoder_states[54], new_encoder_states[55],
                new_encoder_states[56], new_encoder_states[57],
                new_encoder_states[58], new_encoder_states[59],
                new_encoder_states[60], new_encoder_states[61],
                new_encoder_states[62], new_encoder_states[63],
                new_encoder_states[64], new_encoder_states[65],
                new_encoder_states[66], new_encoder_states[67],
                new_encoder_states[68], new_encoder_states[69],
                new_encoder_states[70], new_encoder_states[71],
                new_encoder_states[72], new_encoder_states[73],
                new_encoder_states[74], new_encoder_states[75],
                new_encoder_states[76], new_encoder_states[77],
                new_encoder_states[78], new_encoder_states[79],
                new_encoder_states[80], new_encoder_states[81],
                new_encoder_states[82], new_encoder_states[83],
                new_encoder_states[84], new_encoder_states[85],
                new_encoder_states[86], new_encoder_states[87],
                new_encoder_states[88], new_encoder_states[89],
                new_encoder_states[90], new_encoder_states[91],
                new_encoder_states[92], new_encoder_states[93],
                new_encoder_states[94], new_encoder_states[95],
                new_cached_embed_left_pad,
                new_processed_lens,
                )

# ── Check what methods inner encoder has ──
print("Inner encoder (Zipformer) methods:")
print(list(model.encoder.encoder._c._method_names()))

Inner encoder (Zipformer) methods:
['forward', 'get_init_states', 'get_chunk_info', '_get_full_dim_output', 'streaming_forward']


In [13]:
# Cell 21: Fix — hardcode chunk_size & left_context_len, fix padding mask

import torch
import torch.nn as nn

# From our Cell 5 scraping:
CHUNK_SIZE       = 32
LEFT_CONTEXT_LEN = 128
BATCH_SIZE       = 1
RAW_FRAMES       = 77
FEAT_DIM         = 80

class PatchedEncoderWrapper(nn.Module):
    def __init__(self, encoder_model):
        super().__init__()
        self.encoder_embed = encoder_model.encoder_embed
        self.encoder       = encoder_model.encoder  # inner Zipformer

    def forward(self,
                features, feature_lengths,
                s0,  s1,  s2,  s3,  s4,  s5,  s6,  s7,
                s8,  s9,  s10, s11, s12, s13, s14, s15,
                s16, s17, s18, s19, s20, s21, s22, s23,
                s24, s25, s26, s27, s28, s29, s30, s31,
                s32, s33, s34, s35, s36, s37, s38, s39,
                s40, s41, s42, s43, s44, s45, s46, s47,
                s48, s49, s50, s51, s52, s53, s54, s55,
                s56, s57, s58, s59, s60, s61, s62, s63,
                s64, s65, s66, s67, s68, s69, s70, s71,
                s72, s73, s74, s75, s76, s77, s78, s79,
                s80, s81, s82, s83, s84, s85, s86, s87,
                s88, s89, s90, s91, s92, s93, s94, s95,
                cached_embed_left_pad,
                processed_lens,
                ):

        # ── encoder_embed streaming forward ──
        x, x_lens, new_cached_embed_left_pad = self.encoder_embed.streaming_forward(
            features, feature_lengths, cached_embed_left_pad
        )

        # ── src_key_padding_mask for current chunk ──
        # x shape: [B, chunk_size, 192] — all frames valid for fixed chunk
        # so current chunk mask is all False (nothing masked)
        B = x.shape[0]
        src_key_padding_mask = torch.zeros(B, CHUNK_SIZE, dtype=torch.bool,
                                           device=x.device)

        # ── processed_lens left-context mask ──
        # shape: [B, LEFT_CONTEXT_LEN]
        arange = torch.arange(LEFT_CONTEXT_LEN, device=x.device).unsqueeze(0).expand(B, LEFT_CONTEXT_LEN)
        processed_mask  = torch.le(processed_lens.unsqueeze(1), arange)
        processed_mask0 = torch.flip(processed_mask, [1])

        new_processed_lens = processed_lens + x_lens

        # full mask: [B, LEFT_CONTEXT_LEN + CHUNK_SIZE]
        src_key_padding_mask0 = torch.cat([processed_mask0, src_key_padding_mask], dim=1)

        # ── inner Zipformer streaming_forward ──
        encoder_states = [
            s0,  s1,  s2,  s3,  s4,  s5,  s6,  s7,
            s8,  s9,  s10, s11, s12, s13, s14, s15,
            s16, s17, s18, s19, s20, s21, s22, s23,
            s24, s25, s26, s27, s28, s29, s30, s31,
            s32, s33, s34, s35, s36, s37, s38, s39,
            s40, s41, s42, s43, s44, s45, s46, s47,
            s48, s49, s50, s51, s52, s53, s54, s55,
            s56, s57, s58, s59, s60, s61, s62, s63,
            s64, s65, s66, s67, s68, s69, s70, s71,
            s72, s73, s74, s75, s76, s77, s78, s79,
            s80, s81, s82, s83, s84, s85, s86, s87,
            s88, s89, s90, s91, s92, s93, s94, s95,
        ]

        x0 = torch.permute(x, [1, 0, 2])  # [B,T,C] → [T,B,C]
        encoder_out, encoder_out_lens, new_encoder_states = self.encoder.streaming_forward(
            x0, x_lens, encoder_states, src_key_padding_mask0
        )
        encoder_out0 = torch.permute(encoder_out, [1, 0, 2])  # [T,B,C] → [B,T,C]

        return (encoder_out0, encoder_out_lens,
                new_encoder_states[0],  new_encoder_states[1],
                new_encoder_states[2],  new_encoder_states[3],
                new_encoder_states[4],  new_encoder_states[5],
                new_encoder_states[6],  new_encoder_states[7],
                new_encoder_states[8],  new_encoder_states[9],
                new_encoder_states[10], new_encoder_states[11],
                new_encoder_states[12], new_encoder_states[13],
                new_encoder_states[14], new_encoder_states[15],
                new_encoder_states[16], new_encoder_states[17],
                new_encoder_states[18], new_encoder_states[19],
                new_encoder_states[20], new_encoder_states[21],
                new_encoder_states[22], new_encoder_states[23],
                new_encoder_states[24], new_encoder_states[25],
                new_encoder_states[26], new_encoder_states[27],
                new_encoder_states[28], new_encoder_states[29],
                new_encoder_states[30], new_encoder_states[31],
                new_encoder_states[32], new_encoder_states[33],
                new_encoder_states[34], new_encoder_states[35],
                new_encoder_states[36], new_encoder_states[37],
                new_encoder_states[38], new_encoder_states[39],
                new_encoder_states[40], new_encoder_states[41],
                new_encoder_states[42], new_encoder_states[43],
                new_encoder_states[44], new_encoder_states[45],
                new_encoder_states[46], new_encoder_states[47],
                new_encoder_states[48], new_encoder_states[49],
                new_encoder_states[50], new_encoder_states[51],
                new_encoder_states[52], new_encoder_states[53],
                new_encoder_states[54], new_encoder_states[55],
                new_encoder_states[56], new_encoder_states[57],
                new_encoder_states[58], new_encoder_states[59],
                new_encoder_states[60], new_encoder_states[61],
                new_encoder_states[62], new_encoder_states[63],
                new_encoder_states[64], new_encoder_states[65],
                new_encoder_states[66], new_encoder_states[67],
                new_encoder_states[68], new_encoder_states[69],
                new_encoder_states[70], new_encoder_states[71],
                new_encoder_states[72], new_encoder_states[73],
                new_encoder_states[74], new_encoder_states[75],
                new_encoder_states[76], new_encoder_states[77],
                new_encoder_states[78], new_encoder_states[79],
                new_encoder_states[80], new_encoder_states[81],
                new_encoder_states[82], new_encoder_states[83],
                new_encoder_states[84], new_encoder_states[85],
                new_encoder_states[86], new_encoder_states[87],
                new_encoder_states[88], new_encoder_states[89],
                new_encoder_states[90], new_encoder_states[91],
                new_encoder_states[92], new_encoder_states[93],
                new_encoder_states[94], new_encoder_states[95],
                new_cached_embed_left_pad,
                new_processed_lens,
                )

# ── Prepare inputs ──
features        = torch.randn(BATCH_SIZE, RAW_FRAMES, FEAT_DIM)
feature_lengths = torch.tensor([RAW_FRAMES] * BATCH_SIZE, dtype=torch.int32)
init_states     = model.encoder.get_init_states(batch_size=BATCH_SIZE,
                                                 device=torch.device("cpu"))
encoder_states_96 = init_states[:96]
embed_cache       = init_states[96]
processed_lens    = init_states[97]

wrapper = PatchedEncoderWrapper(model.encoder)
wrapper.eval()

# ── Test forward ──
print("Testing PatchedEncoderWrapper ...")
with torch.no_grad():
    outs = wrapper(features, feature_lengths,
                   *encoder_states_96, embed_cache, processed_lens)
print(f"  outputs           : {len(outs)}")
print(f"  encoder_out       : {outs[0].shape}  expected [1,16,512]")
print(f"  encoder_out_lens  : {outs[1]}")
print(f"  new_state_0       : {outs[2].shape}")
print(f"  new_embed_cache   : {outs[-2].shape}")
print(f"  new_processed_lens: {outs[-1]}")

# ── Compare vs reference ──
print("\nComparing vs model.encoder.forward ...")
with torch.no_grad():
    ref_out, ref_lens, ref_states = model.encoder.forward(
        features, feature_lengths, init_states
    )
diff = (outs[0] - ref_out).abs().max().item()
print(f"  max diff encoder_out: {diff:.8f}  ({'✅ match' if diff < 1e-4 else '❌ mismatch'})")

# ── Trace ──
print("\nTracing ...")
with torch.no_grad():
    traced_patched = torch.jit.trace(
        wrapper,
        (features, feature_lengths,
         *encoder_states_96, embed_cache, processed_lens),
        strict=False,
    )
print("✅ Trace succeeded")

# verify trace matches
with torch.no_grad():
    outs_t = traced_patched(features, feature_lengths,
                             *encoder_states_96, embed_cache, processed_lens)
diff_t = (outs_t[0] - ref_out).abs().max().item()
print(f"  traced max diff: {diff_t:.8f}  ({'✅ match' if diff_t < 1e-4 else '❌ mismatch'})")

Testing PatchedEncoderWrapper ...
  outputs           : 100
  encoder_out       : torch.Size([1, 16, 512])  expected [1,16,512]
  encoder_out_lens  : tensor([16], dtype=torch.int32)
  new_state_0       : torch.Size([128, 1, 128])
  new_embed_cache   : torch.Size([1, 128, 3, 19])
  new_processed_lens: tensor([32], dtype=torch.int32)

Comparing vs model.encoder.forward ...
  max diff encoder_out: 0.00000000  (✅ match)

Tracing ...
✅ Trace succeeded
  traced max diff: 0.00000000  (✅ match)


In [14]:
import torch, torch.nn as nn, onnx, onnxruntime as ort, numpy as np, os, time

PT_PATH = "/kaggle/input/models/nishargnargund/gujarati-asr/pytorch/default/1/SPRING_INX_streaming_k2_Gujarati.pt"
if 'model' not in dir():
    model = torch.jit.load(PT_PATH, map_location="cpu"); model.eval()
    print("Model reloaded")
else:
    print("Model already loaded")

BATCH_SIZE       = 1
FEAT_DIM         = 80
RAW_FRAMES       = 77      # (77-7)//2=35, 35-3=32 = chunk_size
CHUNK_SIZE       = 32      # encoder output T=16
LEFT_CONTEXT_LEN = 128
CONTEXT_SIZE     = 2
VOCAB_SIZE       = 900
ENCODER_DIM      = 512
print(f"chunk_size={model.encoder.chunk_size}, left_context_len={model.encoder.left_context_len}")


Model already loaded
chunk_size=32, left_context_len=128


In [15]:
class PatchedEncoderWrapper(nn.Module):
    """
    Manually replicates StreamingEncoderModel.forward() with all
    dynamic list-indexing done in Python (before tracing).

    The original does states[-2] / states[-1] which ONNX cannot export.
    We pre-extract all three regions as named args instead.
    """
    def __init__(self, encoder_model):
        super().__init__()
        self.encoder_embed = encoder_model.encoder_embed   # Conv2dSubsampling
        self.encoder       = encoder_model.encoder         # inner Zipformer

    def forward(self,
                features, feature_lengths,
                s0,
                s1,
                s2,
                s3,
                s4,
                s5,
                s6,
                s7,
                s8,
                s9,
                s10,
                s11,
                s12,
                s13,
                s14,
                s15,
                s16,
                s17,
                s18,
                s19,
                s20,
                s21,
                s22,
                s23,
                s24,
                s25,
                s26,
                s27,
                s28,
                s29,
                s30,
                s31,
                s32,
                s33,
                s34,
                s35,
                s36,
                s37,
                s38,
                s39,
                s40,
                s41,
                s42,
                s43,
                s44,
                s45,
                s46,
                s47,
                s48,
                s49,
                s50,
                s51,
                s52,
                s53,
                s54,
                s55,
                s56,
                s57,
                s58,
                s59,
                s60,
                s61,
                s62,
                s63,
                s64,
                s65,
                s66,
                s67,
                s68,
                s69,
                s70,
                s71,
                s72,
                s73,
                s74,
                s75,
                s76,
                s77,
                s78,
                s79,
                s80,
                s81,
                s82,
                s83,
                s84,
                s85,
                s86,
                s87,
                s88,
                s89,
                s90,
                s91,
                s92,
                s93,
                s94,
                s95,
                cached_embed_left_pad,   # state[96]: [B,128,3,19] float32
                processed_lens,          # state[97]: [B]          int32
                ):
        # 1. encoder_embed streaming forward
        x, x_lens, new_cached_embed_left_pad = self.encoder_embed.streaming_forward(
            features, feature_lengths, cached_embed_left_pad
        )

        # 2. Padding masks (replicates StreamingEncoderModel.forward internal logic)
        B = x.shape[0]
        src_key_padding_mask = torch.zeros(B, CHUNK_SIZE, dtype=torch.bool, device=x.device)
        arange = torch.arange(LEFT_CONTEXT_LEN, device=x.device).unsqueeze(0).expand(B, LEFT_CONTEXT_LEN)
        processed_mask  = torch.le(processed_lens.unsqueeze(1), arange)
        processed_mask0 = torch.flip(processed_mask, [1])
        new_processed_lens = processed_lens + x_lens
        src_key_padding_mask0 = torch.cat([processed_mask0, src_key_padding_mask], dim=1)

        # 3. Assemble 96 encoder states list in Python (static indexing, not traced)
        encoder_states = [
            s0,
            s1,
            s2,
            s3,
            s4,
            s5,
            s6,
            s7,
            s8,
            s9,
            s10,
            s11,
            s12,
            s13,
            s14,
            s15,
            s16,
            s17,
            s18,
            s19,
            s20,
            s21,
            s22,
            s23,
            s24,
            s25,
            s26,
            s27,
            s28,
            s29,
            s30,
            s31,
            s32,
            s33,
            s34,
            s35,
            s36,
            s37,
            s38,
            s39,
            s40,
            s41,
            s42,
            s43,
            s44,
            s45,
            s46,
            s47,
            s48,
            s49,
            s50,
            s51,
            s52,
            s53,
            s54,
            s55,
            s56,
            s57,
            s58,
            s59,
            s60,
            s61,
            s62,
            s63,
            s64,
            s65,
            s66,
            s67,
            s68,
            s69,
            s70,
            s71,
            s72,
            s73,
            s74,
            s75,
            s76,
            s77,
            s78,
            s79,
            s80,
            s81,
            s82,
            s83,
            s84,
            s85,
            s86,
            s87,
            s88,
            s89,
            s90,
            s91,
            s92,
            s93,
            s94,
            s95,
        ]

        # 4. Inner Zipformer streaming_forward
        x0 = torch.permute(x, [1, 0, 2])           # [B,T,C] -> [T,B,C]
        encoder_out, encoder_out_lens, new_encoder_states = self.encoder.streaming_forward(
            x0, x_lens, encoder_states, src_key_padding_mask0
        )
        encoder_out0 = torch.permute(encoder_out, [1, 0, 2])   # -> [B,T,C]

        return (encoder_out0, encoder_out_lens,
                new_encoder_states[0],
                new_encoder_states[1],
                new_encoder_states[2],
                new_encoder_states[3],
                new_encoder_states[4],
                new_encoder_states[5],
                new_encoder_states[6],
                new_encoder_states[7],
                new_encoder_states[8],
                new_encoder_states[9],
                new_encoder_states[10],
                new_encoder_states[11],
                new_encoder_states[12],
                new_encoder_states[13],
                new_encoder_states[14],
                new_encoder_states[15],
                new_encoder_states[16],
                new_encoder_states[17],
                new_encoder_states[18],
                new_encoder_states[19],
                new_encoder_states[20],
                new_encoder_states[21],
                new_encoder_states[22],
                new_encoder_states[23],
                new_encoder_states[24],
                new_encoder_states[25],
                new_encoder_states[26],
                new_encoder_states[27],
                new_encoder_states[28],
                new_encoder_states[29],
                new_encoder_states[30],
                new_encoder_states[31],
                new_encoder_states[32],
                new_encoder_states[33],
                new_encoder_states[34],
                new_encoder_states[35],
                new_encoder_states[36],
                new_encoder_states[37],
                new_encoder_states[38],
                new_encoder_states[39],
                new_encoder_states[40],
                new_encoder_states[41],
                new_encoder_states[42],
                new_encoder_states[43],
                new_encoder_states[44],
                new_encoder_states[45],
                new_encoder_states[46],
                new_encoder_states[47],
                new_encoder_states[48],
                new_encoder_states[49],
                new_encoder_states[50],
                new_encoder_states[51],
                new_encoder_states[52],
                new_encoder_states[53],
                new_encoder_states[54],
                new_encoder_states[55],
                new_encoder_states[56],
                new_encoder_states[57],
                new_encoder_states[58],
                new_encoder_states[59],
                new_encoder_states[60],
                new_encoder_states[61],
                new_encoder_states[62],
                new_encoder_states[63],
                new_encoder_states[64],
                new_encoder_states[65],
                new_encoder_states[66],
                new_encoder_states[67],
                new_encoder_states[68],
                new_encoder_states[69],
                new_encoder_states[70],
                new_encoder_states[71],
                new_encoder_states[72],
                new_encoder_states[73],
                new_encoder_states[74],
                new_encoder_states[75],
                new_encoder_states[76],
                new_encoder_states[77],
                new_encoder_states[78],
                new_encoder_states[79],
                new_encoder_states[80],
                new_encoder_states[81],
                new_encoder_states[82],
                new_encoder_states[83],
                new_encoder_states[84],
                new_encoder_states[85],
                new_encoder_states[86],
                new_encoder_states[87],
                new_encoder_states[88],
                new_encoder_states[89],
                new_encoder_states[90],
                new_encoder_states[91],
                new_encoder_states[92],
                new_encoder_states[93],
                new_encoder_states[94],
                new_encoder_states[95],
                new_cached_embed_left_pad,
                new_processed_lens,
                )

print("PatchedEncoderWrapper defined")


PatchedEncoderWrapper defined


In [16]:
init_states       = model.encoder.get_init_states(batch_size=BATCH_SIZE, device=torch.device("cpu"))
encoder_states_96 = init_states[:96]
embed_cache       = init_states[96]    # [1, 128, 3, 19] float32
processed_lens    = init_states[97]    # [1]              int32

features        = torch.randn(BATCH_SIZE, RAW_FRAMES, FEAT_DIM)
feature_lengths = torch.tensor([RAW_FRAMES] * BATCH_SIZE, dtype=torch.int32)

with torch.no_grad():
    ref_out, ref_lens, ref_states = model.encoder.forward(features, feature_lengths, init_states)
print(f"Reference encoder_out: {ref_out.shape}")

wrapper = PatchedEncoderWrapper(model.encoder)
wrapper.eval()

with torch.no_grad():
    outs = wrapper(features, feature_lengths, *encoder_states_96, embed_cache, processed_lens)

diff = (outs[0] - ref_out).abs().max().item()
print(f"PatchedWrapper vs reference: max_diff={diff:.2e}  {'pass' if diff < 1e-4 else 'FAIL'}")
assert diff < 1e-4
print(f"Total outputs: {len(outs)}  (expected 100)")


Reference encoder_out: torch.Size([1, 16, 512])
PatchedWrapper vs reference: max_diff=0.00e+00  pass
Total outputs: 100  (expected 100)


In [17]:
print("Tracing ...")
with torch.no_grad():
    traced_encoder = torch.jit.trace(
        wrapper,
        (features, feature_lengths, *encoder_states_96, embed_cache, processed_lens),
        strict=False,
    )
with torch.no_grad():
    outs_t = traced_encoder(features, feature_lengths, *encoder_states_96, embed_cache, processed_lens)
diff_t = (outs_t[0] - ref_out).abs().max().item()
print(f"Traced vs reference: max_diff={diff_t:.2e}  {'pass' if diff_t < 1e-4 else 'FAIL'}")
assert diff_t < 1e-4
print("Trace succeeded")


Tracing ...
Traced vs reference: max_diff=0.00e+00  pass
Trace succeeded


In [18]:
# Cell E-SURGERY: Graph surgery on traced_encoder to remove dynamic prim::Loop
#
# Root cause chain:
#   jit.trace inlines Zipformer → prim::CallMethod in graph
#   Legacy ONNX exporter calls _jit_pass_inline → prim::Loop exposed
#   _jit_pass_lower_all_tuples hits aten::__getitem__ with variable index → CRASH
#   TS2EPConverter also fails on prim::Loop → CRASH
#
# Fix: do the inline+unroll OURSELVES before export.
#   After unrolling, remaining loops have constant bounds → exporter succeeds.

print("Applying graph surgery to traced_encoder ...")
g = traced_encoder.graph

# Step 1: inline all prim::CallMethod → exposes Zipformer's internal loops
torch._C._jit_pass_inline(g)
n_after_inline = sum(1 for n in g.nodes() if n.kind() == 'prim::Loop')
print(f"  After inline : {n_after_inline} prim::Loop nodes now visible")

# Step 2: unroll loops (factor-8 + constant-bound remainder)
torch._C._jit_pass_loop_unrolling(g)
n_after_unroll = sum(1 for n in g.nodes() if n.kind() == 'prim::Loop')
print(f"  After unroll : {n_after_unroll} prim::Loop nodes remain (constant-bound remainder)")

# Quick sanity check — traced_encoder should still produce correct output
with torch.no_grad():
    test_outs = traced_encoder(features, feature_lengths, *encoder_states_96, embed_cache, processed_lens)
diff = (test_outs[0] - ref_out).abs().max().item()
print(f"  Output check : max_diff={diff:.2e}  {'OK' if diff < 1e-4 else 'MISMATCH'}")
assert diff < 1e-4, "Graph surgery broke the model!"
print("Graph surgery complete.")# Cell E-SURGERY: Graph surgery on traced_encoder to remove dynamic prim::Loop
#
# Root cause chain:
#   jit.trace inlines Zipformer → prim::CallMethod in graph
#   Legacy ONNX exporter calls _jit_pass_inline → prim::Loop exposed
#   _jit_pass_lower_all_tuples hits aten::__getitem__ with variable index → CRASH
#   TS2EPConverter also fails on prim::Loop → CRASH
#
# Fix: do the inline+unroll OURSELVES before export.
#   After unrolling, remaining loops have constant bounds → exporter succeeds.

print("Applying graph surgery to traced_encoder ...")
g = traced_encoder.graph

# Step 1: inline all prim::CallMethod → exposes Zipformer's internal loops
torch._C._jit_pass_inline(g)
n_after_inline = sum(1 for n in g.nodes() if n.kind() == 'prim::Loop')
print(f"  After inline : {n_after_inline} prim::Loop nodes now visible")

# Step 2: unroll loops (factor-8 + constant-bound remainder)
torch._C._jit_pass_loop_unrolling(g)
n_after_unroll = sum(1 for n in g.nodes() if n.kind() == 'prim::Loop')
print(f"  After unroll : {n_after_unroll} prim::Loop nodes remain (constant-bound remainder)")

# Quick sanity check — traced_encoder should still produce correct output
with torch.no_grad():
    test_outs = traced_encoder(features, feature_lengths, *encoder_states_96, embed_cache, processed_lens)
diff = (test_outs[0] - ref_out).abs().max().item()
print(f"  Output check : max_diff={diff:.2e}  {'OK' if diff < 1e-4 else 'MISMATCH'}")
assert diff < 1e-4, "Graph surgery broke the model!"
print("Graph surgery complete.")

Applying graph surgery to traced_encoder ...
  After inline : 18 prim::Loop nodes now visible
  After unroll : 34 prim::Loop nodes remain (constant-bound remainder)
  Output check : max_diff=0.00e+00  OK
Graph surgery complete.
Applying graph surgery to traced_encoder ...
  After inline : 34 prim::Loop nodes now visible
  After unroll : 68 prim::Loop nodes remain (constant-bound remainder)
  Output check : max_diff=0.00e+00  OK
Graph surgery complete.


In [19]:
# Cell E-SURGERY (corrected): 4-step graph surgery on traced_encoder
#
# Problem sequence in legacy ONNX exporter:
#   _jit_pass_inline (inlines Zipformer's TorchScript)
#   → prim::Loop with bound = aten::len(states_list)  ← dynamic
#   → _jit_pass_lower_all_tuples crashes on dynamic tuple/list index
#
# Our fix: run the SAME inline FIRST ourselves, then:
#   peephole + constant_propagation → folds aten::len(ListConstruct{96}) → Constant[96]
#   constant_loop_unrolling → fully unrolls the now-constant-bound loop
#   Result: zero loops with dynamic indices; exporter's own inline is then a no-op

print("Applying 4-step graph surgery to traced_encoder ...")
g = traced_encoder.graph

# Step 1: inline — same pass the ONNX exporter would run; do it ourselves first
torch._C._jit_pass_inline(g)
n1 = sum(1 for n in g.nodes() if n.kind() == 'prim::Loop')
print(f"  After inline:       {n1} prim::Loop (bound = aten::len, still dynamic)")

# Step 2: peephole + constant propagation
#   aten::len(prim::ListConstruct{96 inputs}) → prim::Constant[value=96]
torch._C._jit_pass_peephole(g, True)
torch._C._jit_pass_constant_propagation(g)
n2 = sum(1 for n in g.nodes() if n.kind() == 'prim::Loop')
all_const = all(
    list(n.inputs())[0].node().kind() == 'prim::Constant'
    for n in g.nodes() if n.kind() == 'prim::Loop'
)
print(f"  After constprop:    {n2} prim::Loop (bounds now constant: {all_const})")

# Step 3: constant loop unrolling — now fully unrolls since bounds are constant ints
torch._C._jit_pass_constant_loop_unrolling(g)
n3 = sum(1 for n in g.nodes() if n.kind() == 'prim::Loop')
print(f"  After const_unroll: {n3} prim::Loop")

# Step 4: cleanup
torch._C._jit_pass_dce(g)

# Verify traced_encoder still produces correct output after surgery
with torch.no_grad():
    test_outs = traced_encoder(features, feature_lengths, *encoder_states_96, embed_cache, processed_lens)
diff = (test_outs[0] - ref_out).abs().max().item()
print(f"  Output check:       max_diff={diff:.2e}  {'OK' if diff < 1e-5 else 'MISMATCH'}")
assert diff < 1e-5, "Graph surgery broke the model!"
print("Graph surgery complete.")

Applying 4-step graph surgery to traced_encoder ...
  After inline:       68 prim::Loop (bound = aten::len, still dynamic)
  After constprop:    68 prim::Loop (bounds now constant: False)
  After const_unroll: 68 prim::Loop
  Output check:       max_diff=0.00e+00  OK
Graph surgery complete.


In [20]:
g = traced_encoder.graph

loop_info = []
for node in g.nodes():
    if node.kind() == 'prim::Loop':
        inputs = list(node.inputs())
        bound = inputs[0]
        bound_kind = bound.node().kind()
        block = list(node.blocks())[0]
        inner_kinds = [n.kind() for n in block.nodes()]
        loop_info.append({
            'bound_kind': bound_kind,
            'bound_node': str(bound.node()),
            'inner_ops': inner_kinds[:5],
        })

from collections import Counter
bound_counts = Counter(x['bound_kind'] for x in loop_info)
print(f"Loop bound types: {dict(bound_counts)}")

print("\nNon-constant bound loops (first 5):")
shown = 0
for info in loop_info:
    if info['bound_kind'] != 'prim::Constant':
        print(f"  bound: {info['bound_node'].strip()}")
        print(f"  inner: {info['inner_ops']}")
        print()
        shown += 1
        if shown >= 5:
            break

Loop bound types: {'aten::__round_to_zero_floordiv': 34, 'aten::sub': 34}

Non-constant bound loops (first 5):
  bound: %19665 : int = aten::__round_to_zero_floordiv(%19154, %19664)
  inner: ['aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze']

  bound: %19669 : int = aten::sub(%19154, %19668)
  inner: ['aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze']

  bound: %19685 : int = aten::__round_to_zero_floordiv(%19158, %19684)
  inner: ['aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze']

  bound: %19689 : int = aten::sub(%19158, %19688)
  inner: ['aten::unsqueeze']

  bound: %19768 : int = aten::__round_to_zero_floordiv(%19174, %19767)
  inner: ['aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze', 'aten::unsqueeze']



In [21]:
g = traced_encoder.graph

shown = 0
for node in g.nodes():
    if node.kind() == 'prim::Loop':
        inputs = list(node.inputs())
        bound = inputs[0]
        if bound.node().kind() != 'prim::Constant':
            def trace_val(v, depth=0):
                n = v.node()
                prefix = "  " * depth
                print(f"{prefix}{n.kind()} -> {str(v)[:80].strip()}")
                if depth < 4 and n.kind() not in ('prim::Param', 'prim::Constant'):
                    for inp in list(n.inputs())[:2]:
                        trace_val(inp, depth+1)
            
            print("Loop bound trace:")
            trace_val(bound)
            print()
            shown += 1
            if shown >= 3:
                break

Loop bound trace:
aten::__round_to_zero_floordiv -> 19665 defined in (%19665 : int = aten::__round_to_zero_floordiv(%19154, %19664)
  aten::__round_to_zero_floordiv -> 19154 defined in (%19154 : int = aten::__round_to_zero_floordiv(%7032, %19153)
)
    aten::__range_length -> 7032 defined in (%7032 : int = aten::__range_length(%7030, %7031, %6886) # /nlsa
      aten::add -> 7030 defined in (%7030 : int = aten::add(%channel_dim1.7, %6886) # /nlsasfs/home
        prim::If -> channel_dim1.7 defined in (%channel_dim1.7 : int = prim::If(%7025) # /nlsasfs/ho
        prim::Constant -> 6886 defined in (%6886 : int = prim::Constant[value=1]() # /nlsasfs/home/nltm-pi
      aten::dim -> 7031 defined in (%7031 : int = aten::dim(%x12.21) # <string>:3:9
)
        aten::linear -> x12.21 defined in (%x12.21 : Tensor = aten::linear(%x11.8, %weight.152, %bias.12
    prim::Constant -> 19153 defined in (%19153 : int = prim::Constant[value=8]()
)
  prim::Constant -> 19664 defined in (%19664 : int = prim::C

In [22]:
g = traced_encoder.graph

print("=== aten::dim nodes ===")
for node in g.nodes():
    if node.kind() == 'aten::dim':
        inp = list(node.inputs())[0]
        inp_type = inp.type()
        try:
            sizes = inp_type.sizes()
            print(f"  tensor sizes: {sizes}  -> rank={len(sizes)}")
        except:
            print(f"  tensor type: {inp_type}  (no static sizes)")

print("\n=== aten::__range_length nodes (first 3) ===")
count = 0
for node in g.nodes():
    if node.kind() == 'aten::__range_length':
        inputs = list(node.inputs())
        print(f"  args: {[str(i)[:60].strip() for i in inputs]}")
        count += 1
        if count >= 3: break

=== aten::dim nodes ===
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)
  tensor type: Tensor  (no static sizes)

=== aten::__range_length nodes (first 3) ===
  args: ['7030 defined in (%7030 : int = aten::add(%channel_dim1.7, %6', '7031 defined in (%7031 : int = aten::dim(%x12.21) # <string>', '6886 defined in (%6886 : int = prim::Constant[value=1]() # /']
  args: ['7876 defined in (%7876

In [23]:
g = traced_encoder.graph

def find_all_nodes_recursive(block, kind):
    results = []
    for node in block.nodes():
        if node.kind() == kind:
            results.append(node)
        for b in node.blocks():
            results.extend(find_all_nodes_recursive(b, kind))
    return results

# Search recursively including inside loops
ti_nodes = find_all_nodes_recursive(g, 'prim::TupleIndex')
print(f"prim::TupleIndex nodes: {len(ti_nodes)}")
for tn in ti_nodes[:5]:
    inputs = list(tn.inputs())
    tup, idx = inputs[0], inputs[1]
    print(f"  tuple type: {tup.type()}")
    print(f"  index node: {idx.node().kind()} = {str(idx.node())[:80]}")
    print()

getitem_nodes = find_all_nodes_recursive(g, 'aten::__getitem__')
print(f"\naten::__getitem__ nodes total: {len(getitem_nodes)}")
tuple_getitems = []
for gn in getitem_nodes:
    inp = list(gn.inputs())[0]
    if 'Tuple' in str(inp.type()) or '(' in str(inp.type()):
        tuple_getitems.append(gn)
print(f"aten::__getitem__ on tuple-like: {len(tuple_getitems)}")
for gn in tuple_getitems[:5]:
    inputs = list(gn.inputs())
    tup, idx = inputs[0], inputs[1]
    print(f"  type: {tup.type()}")
    print(f"  index: {idx.node().kind()} = {str(idx.node())[:80]}")
    print()

prim::TupleIndex nodes: 30
  tuple type: Tuple[int, int]
  index node: prim::Constant = %6884 : int = prim::Constant[value=0]() # /nlsasfs/home/nltm-pilot/msdafini/othe

  tuple type: Tuple[int, int]
  index node: prim::Constant = %6884 : int = prim::Constant[value=0]() # /nlsasfs/home/nltm-pilot/msdafini/othe

  tuple type: Tuple[int, int]
  index node: prim::Constant = %6884 : int = prim::Constant[value=0]() # /nlsasfs/home/nltm-pilot/msdafini/othe

  tuple type: Tuple[int, int]
  index node: prim::Constant = %6884 : int = prim::Constant[value=0]() # /nlsasfs/home/nltm-pilot/msdafini/othe

  tuple type: Tuple[int, int]
  index node: prim::Constant = %6886 : int = prim::Constant[value=1]() # /nlsasfs/home/nltm-pilot/msdafini/othe


aten::__getitem__ nodes total: 406
aten::__getitem__ on tuple-like: 0


In [24]:
# Cell: Shape Audit — run each submodule with dummy inputs to capture exact I/O shapes

import torch

model.eval()

# ── 1. encoder_embed (Conv2dSubsampling) ────────────────────────────────────
# Standard input: (batch, time, fbank_features)
# SPRING_INX uses 80-dim fbank
batch, time_frames, n_mels = 1, 100, 80

x = torch.randn(batch, time_frames, n_mels)
x_lens = torch.tensor([time_frames], dtype=torch.int64)

print("=" * 60)
print("encoder_embed (Conv2dSubsampling)")
print("=" * 60)
print(f"  INPUT  x:      {x.shape}")
print(f"  INPUT  x_lens: {x_lens.shape}")

try:
    methods = list(model.encoder_embed._c._method_names())
    print(f"  Methods: {methods}")
    
    # Try forward
    out = model.encoder_embed.forward(x, x_lens)
    if isinstance(out, tuple):
        print(f"  OUTPUT is tuple of {len(out)} elements:")
        for i, o in enumerate(out):
            if isinstance(o, torch.Tensor):
                print(f"    [{i}] Tensor: {o.shape} dtype={o.dtype}")
            else:
                print(f"    [{i}] {type(o).__name__}: {o}")
    else:
        print(f"  OUTPUT: {out.shape}")
except Exception as e:
    print(f"  forward() failed: {e}")

# Try without lens
try:
    out2 = model.encoder_embed.forward(x)
    print(f"  OUTPUT (no lens): {out2.shape if isinstance(out2, torch.Tensor) else type(out2)}")
except Exception as e:
    print(f"  forward(x only) failed: {e}")

encoder_embed (Conv2dSubsampling)
  INPUT  x:      torch.Size([1, 100, 80])
  INPUT  x_lens: torch.Size([1])
  Methods: ['forward', 'get_init_states', 'streaming_forward']
  OUTPUT is tuple of 2 elements:
    [0] Tensor: torch.Size([1, 46, 192]) dtype=torch.float32
    [1] Tensor: torch.Size([1]) dtype=torch.int64
  forward(x only) failed: forward() is missing value for argument 'x_lens'. Declaration: forward(__torch__.subsampling.Conv2dSubsampling self, Tensor x, Tensor x_lens) -> ((Tensor, Tensor))


In [25]:
# Cell: Audit encoder (StreamingEncoderModel)

import torch

model.eval()

# encoder_embed output feeds directly into encoder
# From previous: output shape = (1, 46, 192)
batch = 1
enc_time = 46
enc_dim = 192

x_enc = torch.randn(batch, enc_time, enc_dim)
x_enc_lens = torch.tensor([enc_time], dtype=torch.int64)

print("=" * 60)
print("encoder (StreamingEncoderModel)")
print("=" * 60)

# 1. List methods
methods = list(model.encoder._c._method_names())
print(f"  Methods: {methods}")

# 2. Try get_init_states first — streaming models need states
print("\n--- get_init_states ---")
try:
    states = model.encoder.get_init_states(batch_size=batch)
    if isinstance(states, (list, tuple)):
        print(f"  Returns {type(states).__name__} of {len(states)} elements:")
        for i, s in enumerate(states):
            if isinstance(s, torch.Tensor):
                print(f"    [{i}] Tensor: {s.shape} dtype={s.dtype}")
            elif isinstance(s, (list, tuple)):
                print(f"    [{i}] {type(s).__name__} of {len(s)}:")
                for j, ss in enumerate(s):
                    if isinstance(ss, torch.Tensor):
                        print(f"      [{j}] Tensor: {ss.shape} dtype={ss.dtype}")
            else:
                print(f"    [{i}] {type(s).__name__}: {s}")
    else:
        print(f"  Returns: {type(states)}")
except Exception as e:
    print(f"  get_init_states failed: {e}")
    states = None

# 3. Try forward
print("\n--- forward ---")
try:
    out = model.encoder.forward(x_enc, x_enc_lens)
    if isinstance(out, tuple):
        print(f"  OUTPUT tuple of {len(out)}:")
        for i, o in enumerate(out):
            if isinstance(o, torch.Tensor):
                print(f"    [{i}] Tensor: {o.shape} dtype={o.dtype}")
            elif isinstance(o, (list, tuple)):
                print(f"    [{i}] {type(o).__name__} of {len(o)}:")
                for j, oo in enumerate(o):
                    if isinstance(oo, torch.Tensor):
                        print(f"      [{j}] Tensor: {oo.shape}")
            else:
                print(f"    [{i}] {type(o).__name__}: {o}")
    else:
        print(f"  OUTPUT: {out.shape}")
except Exception as e:
    print(f"  forward(x, lens) failed: {e}")

# 4. Try streaming_forward if it exists
if 'streaming_forward' in methods:
    print("\n--- streaming_forward ---")
    try:
        # Try with states=None first
        out_s = model.encoder.streaming_forward(x_enc, x_enc_lens, states)
        if isinstance(out_s, tuple):
            print(f"  OUTPUT tuple of {len(out_s)}:")
            for i, o in enumerate(out_s):
                if isinstance(o, torch.Tensor):
                    print(f"    [{i}] Tensor: {o.shape} dtype={o.dtype}")
                elif isinstance(o, (list, tuple)):
                    print(f"    [{i}] {type(o).__name__} of {len(o)}:")
                    for j, oo in enumerate(o):
                        if isinstance(oo, torch.Tensor):
                            print(f"      [{j}] Tensor: {oo.shape}")
                else:
                    print(f"    [{i}] {type(o).__name__}: {o}")
        else:
            print(f"  OUTPUT: {out_s.shape}")
    except Exception as e:
        print(f"  streaming_forward failed: {e}")

encoder (StreamingEncoderModel)
  Methods: ['forward', 'get_init_states']

--- get_init_states ---
  Returns list of 98 elements:
    [0] Tensor: torch.Size([128, 1, 128]) dtype=torch.float32
    [1] Tensor: torch.Size([1, 1, 128, 144]) dtype=torch.float32
    [2] Tensor: torch.Size([128, 1, 48]) dtype=torch.float32
    [3] Tensor: torch.Size([128, 1, 48]) dtype=torch.float32
    [4] Tensor: torch.Size([1, 192, 15]) dtype=torch.float32
    [5] Tensor: torch.Size([1, 192, 15]) dtype=torch.float32
    [6] Tensor: torch.Size([128, 1, 128]) dtype=torch.float32
    [7] Tensor: torch.Size([1, 1, 128, 144]) dtype=torch.float32
    [8] Tensor: torch.Size([128, 1, 48]) dtype=torch.float32
    [9] Tensor: torch.Size([128, 1, 48]) dtype=torch.float32
    [10] Tensor: torch.Size([1, 192, 15]) dtype=torch.float32
    [11] Tensor: torch.Size([1, 192, 15]) dtype=torch.float32
    [12] Tensor: torch.Size([64, 1, 128]) dtype=torch.float32
    [13] Tensor: torch.Size([1, 1, 64, 192]) dtype=torch.float32

In [26]:
# Cell: Run encoder forward, then audit decoder and joiner

import torch

model.eval()

# ── Setup from previous cells ─────────────────────────────
batch = 1
enc_time = 46
enc_dim = 192

x_enc = torch.randn(batch, enc_time, enc_dim)
x_enc_lens = torch.tensor([enc_time], dtype=torch.int64)

# Get init states
states = model.encoder.get_init_states(batch_size=batch)
print(f"States: list of {len(states)} tensors")

# ── 1. Run encoder.forward ────────────────────────────────
print("\n" + "="*60)
print("encoder.forward output")
print("="*60)
try:
    enc_out, enc_out_lens, new_states = model.encoder.forward(x_enc, x_enc_lens, states)
    print(f"  enc_out shape:      {enc_out.shape}  dtype={enc_out.dtype}")
    print(f"  enc_out_lens shape: {enc_out_lens.shape}  value={enc_out_lens}")
    print(f"  new_states: list of {len(new_states)} tensors")
    # Verify state shapes are same as init (they must be for stateful ONNX)
    shape_match = all(new_states[i].shape == states[i].shape for i in range(len(states)))
    print(f"  State shapes preserved: {shape_match}")
    if not shape_match:
        for i in range(len(states)):
            if new_states[i].shape != states[i].shape:
                print(f"    MISMATCH [{i}]: init={states[i].shape} -> out={new_states[i].shape}")
except Exception as e:
    print(f"  encoder.forward failed: {e}")
    enc_out = torch.randn(batch, enc_time, 512)  # fallback for downstream
    enc_out_lens = x_enc_lens

# ── 2. simple_am_proj ─────────────────────────────────────
print("\n" + "="*60)
print("simple_am_proj (Linear)")
print("="*60)
try:
    am_out = model.simple_am_proj.forward(enc_out)
    print(f"  INPUT:  {enc_out.shape}")
    print(f"  OUTPUT: {am_out.shape}")
except Exception as e:
    print(f"  simple_am_proj failed: {e}")
    am_out = enc_out

# ── 3. decoder (Prediction Network) ──────────────────────
print("\n" + "="*60)
print("decoder (Decoder)")
print("="*60)
dec_methods = list(model.decoder._c._method_names())
print(f"  Methods: {dec_methods}")

# Decoder input: token IDs, shape (B, context_size) typically context_size=2
# Start with BOS token = 0
for ctx_size in [1, 2, 3]:
    try:
        y = torch.zeros(batch, ctx_size, dtype=torch.int64)
        dec_out = model.decoder.forward(y, need_pad=torch.tensor(True))
        print(f"  INPUT y shape ({ctx_size}): {y.shape} -> OUTPUT: {dec_out.shape}")
        break
    except Exception as e:
        print(f"  ctx_size={ctx_size} failed: {e}")

# Try without need_pad
try:
    y = torch.zeros(batch, ctx_size, dtype=torch.int64)
    dec_out2 = model.decoder.forward(y, need_pad=torch.tensor(False))
    print(f"  need_pad=False OUTPUT: {dec_out2.shape}")
except Exception as e:
    print(f"  need_pad=False failed: {e}")

# ── 4. simple_lm_proj ─────────────────────────────────────
print("\n" + "="*60)
print("simple_lm_proj (Linear)")
print("="*60)
try:
    lm_out = model.simple_lm_proj.forward(dec_out)
    print(f"  INPUT:  {dec_out.shape}")
    print(f"  OUTPUT: {lm_out.shape}")
except Exception as e:
    print(f"  simple_lm_proj failed: {e}")
    lm_out = dec_out

# ── 5. joiner ─────────────────────────────────────────────
print("\n" + "="*60)
print("joiner (Joiner)")
print("="*60)
joiner_methods = list(model.joiner._c._method_names())
print(f"  Methods: {joiner_methods}")

try:
    # joiner takes am_out + lm_out, must be same time dimension
    # Typical: am=(B,T,C), lm=(B,U,C) -> broadcast to (B,T,U,C)
    # Try squeezing to single frame first
    am_frame = am_out[:, :1, :]   # (B, 1, C)
    lm_frame = lm_out[:, :1, :]   # (B, 1, C)
    j_out = model.joiner.forward(am_frame, lm_frame)
    print(f"  INPUT am: {am_frame.shape}, lm: {lm_frame.shape}")
    print(f"  OUTPUT:   {j_out.shape}")
except Exception as e:
    print(f"  joiner single frame failed: {e}")
    try:
        # Try without frame dim squeeze
        j_out = model.joiner.forward(am_out, lm_out)
        print(f"  INPUT am: {am_out.shape}, lm: {lm_out.shape}")
        print(f"  OUTPUT:   {j_out.shape}")
    except Exception as e2:
        print(f"  joiner full failed: {e2}")

States: list of 98 tensors

encoder.forward output
  encoder.forward failed: The following operation failed in the TorchScript interpreter.
Traceback of TorchScript, serialized code (most recent call last):
  File "code/__torch__.py", line 20, in forward
    cached_embed_left_pad = states[-2]
    encoder_embed = self.encoder_embed
    _1 = (encoder_embed).streaming_forward(features, feature_lengths, cached_embed_left_pad, )
          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ <--- HERE
    x, x_lens, new_cached_embed_left_pad, = _1
    _2 = torch.eq(torch.size(x, 1), chunk_size)
  File "code/__torch__/subsampling.py", line 59, in streaming_forward
    x9 = (conv).forward(x8, )
    convnext = self.convnext
    _4 = (convnext).streaming_forward(x9, cached_left_pad, )
          ~~~~~~~~~~~~~~~~~~~~~~~~~~~ <--- HERE
    x10, cached_left_pad0, = _4
    b, c, t, f, = torch.size(x10)
  File "code/__torch__/subsampling.py", line 144, in streaming_forward
      _13 = torch.add("AssertionError: ", str(_12

In [27]:
# Cell: Discover chunk_size and correct encoder forward call

import torch

model.eval()

# ── Step 1: Read encoder's code to find chunk_size ──────────────────────────
print("=== encoder code ===")
try:
    print(model.encoder.code)
except Exception as e:
    print(f"  .code error: {e}")

print("\n=== encoder_embed.streaming_forward code ===")
try:
    print(model.encoder_embed.streaming_forward.code)
except Exception as e:
    print(f"  error: {e}")

=== encoder code ===
def forward(self,
    features: Tensor,
    feature_lengths: Tensor,
    states: List[Tensor]) -> Tuple[Tensor, Tensor, List[Tensor]]:
  chunk_size = self.chunk_size
  left_context_len = self.left_context_len
  cached_embed_left_pad = states[-2]
  encoder_embed = self.encoder_embed
  _0 = (encoder_embed).streaming_forward(features, feature_lengths, cached_embed_left_pad, )
  x, x_lens, new_cached_embed_left_pad, = _0
  _1 = torch.eq(torch.size(x, 1), chunk_size)
  if _1:
    pass
  else:
    _2 = torch.add("AssertionError: ", str((torch.size(x, 1), chunk_size)))
    ops.prim.RaiseException(_2)
  src_key_padding_mask = __torch__.icefall.utils.make_pad_mask(x_lens, 0, )
  _3 = torch.arange(left_context_len, dtype=None, layout=None, device=ops.prim.device(x))
  processed_mask = torch.expand(_3, [torch.size(x, 0), left_context_len])
  processed_lens = states[-1]
  _4 = torch.le(torch.unsqueeze(processed_lens, 1), processed_mask)
  processed_mask0 = torch.flip(_4, [1])


In [28]:
# Cell: Find chunk_size from state shape and probe encoder with correct chunk

import torch

model.eval()

# From state[-2] = shape [1, 128, 3, 19]
# This is cached_embed_left_pad after Conv2dSubsampling
# Shape: (B, C, left_pad_frames, freq_after_conv)
# freq_after_conv = 19 -> this tells us what the conv output freq dim is
# The encoder expects x chunks that match this freq

# State[-1] = shape [1] int32 -> likely a frame counter / offset

# Let's check what chunk_size the encoder was designed for
# by inspecting the graph constants
print("=== Checking encoder graph for chunk_size constant ===")
enc_methods = list(model.encoder._c._method_names())
print(f"Methods: {enc_methods}")

try:
    g = model.encoder.forward.graph
    chunk_constants = []
    for node in g.nodes():
        if node.kind() == 'prim::Constant':
            val = node.output().toIValue()
            if isinstance(val, int) and 4 <= val <= 128:
                chunk_constants.append(val)
    # Deduplicate and sort
    chunk_constants = sorted(set(chunk_constants))
    print(f"Integer constants in encoder graph (4-128): {chunk_constants[:30]}")
except Exception as e:
    print(f"Graph inspection failed: {e}")

# ── The key insight: encoder.forward() calls encoder_embed.streaming_forward
# internally, so we should NOT feed encoder_embed output into encoder.
# Instead, encoder.forward takes RAW features (B, T, 80)
# Let's try with a chunk that matches the expected size.
# Common Zipformer chunk sizes: 8, 16, 32 frames

print("\n=== Probing encoder.forward with raw features + correct chunk ===")
batch = 1
states = model.encoder.get_init_states(batch_size=batch)

for T in [8, 16, 24, 32, 48, 64]:
    try:
        x_raw = torch.randn(batch, T, 80)          # raw fbank
        x_lens = torch.tensor([T], dtype=torch.int64)
        enc_out, enc_lens, new_states = model.encoder.forward(x_raw, x_lens, states)
        print(f"  T={T:3d} -> enc_out: {enc_out.shape}, enc_lens: {enc_lens.tolist()}, states_preserved: {all(new_states[i].shape == states[i].shape for i in range(len(states)))}")
        break  # found working chunk size
    except Exception as e:
        # Extract key part of error
        err = str(e).split('\n')[0][:120]
        print(f"  T={T:3d} -> FAIL: {err}")

=== Checking encoder graph for chunk_size constant ===
Methods: ['forward', 'get_init_states']
Integer constants in encoder graph (4-128): []

=== Probing encoder.forward with raw features + correct chunk ===
  T=  8 -> FAIL: The following operation failed in the TorchScript interpreter.
  T= 16 -> FAIL: The following operation failed in the TorchScript interpreter.
  T= 24 -> FAIL: The following operation failed in the TorchScript interpreter.
  T= 32 -> FAIL: The following operation failed in the TorchScript interpreter.
  T= 48 -> FAIL: The following operation failed in the TorchScript interpreter.
  T= 64 -> FAIL: The following operation failed in the TorchScript interpreter.


In [29]:
# Cell: Read chunk_size and left_context_len from model attributes, then find valid T

import torch

model.eval()

# ── 1. Read stored attributes ────────────────────────────────────────────────
print("=== encoder attributes ===")
try:
    chunk_size = model.encoder.chunk_size
    print(f"  chunk_size:       {chunk_size}")
except Exception as e:
    print(f"  chunk_size failed: {e}")
    chunk_size = None

try:
    left_context_len = model.encoder.left_context_len
    print(f"  left_context_len: {left_context_len}")
except Exception as e:
    print(f"  left_context_len failed: {e}")

# ── 2. encoder_embed attributes ──────────────────────────────────────────────
print("\n=== encoder_embed attributes ===")
for attr in ['chunk_size', 'left_context_len', 'subsampling_factor']:
    try:
        val = getattr(model.encoder_embed, attr)
        print(f"  {attr}: {val}")
    except:
        pass

# Read convnext padding directly
try:
    pad = model.encoder_embed.convnext.padding
    print(f"  convnext.padding: {pad}")
except Exception as e:
    print(f"  convnext.padding: {e}")

# ── 3. Math: given chunk_size (output frames), find required input T ─────────
# Formula from code: x_lens_out = (x_lens_in - 7) // 2 - 3
# So: chunk_size = (T_in - 7) // 2 - 3
# => T_in = chunk_size * 2 + 13   (minimum that produces exactly chunk_size output frames)
if chunk_size is not None:
    T_required = chunk_size * 2 + 13
    print(f"\n=== Required input T ===")
    print(f"  chunk_size={chunk_size} => T_in needed = {T_required}")
    
    # Verify the formula
    for T in range(T_required - 2, T_required + 4):
        out_frames = (T - 7) // 2 - 3
        print(f"  T={T} -> output_frames={out_frames}  {'✅ MATCH' if out_frames == chunk_size else ''}")

# ── 4. Now try encoder.forward with the correct T ───────────────────────────
print("\n=== encoder.forward with correct T ===")
batch = 1
states = model.encoder.get_init_states(batch_size=batch)

if chunk_size is not None:
    T_in = chunk_size * 2 + 13
    x_raw = torch.randn(batch, T_in, 80)
    x_lens = torch.tensor([T_in], dtype=torch.int64)
    try:
        enc_out, enc_lens, new_states = model.encoder.forward(x_raw, x_lens, states)
        print(f"  INPUT:  (B={batch}, T={T_in}, 80)")
        print(f"  OUTPUT enc_out:  {enc_out.shape}")
        print(f"  OUTPUT enc_lens: {enc_lens.tolist()}")
        print(f"  States preserved: {all(new_states[i].shape == states[i].shape for i in range(len(states)))}")
        # Check output dim for joiner
        print(f"\n  ✅ Encoder output dim = {enc_out.shape[-1]} (this feeds joiner)")
    except Exception as e:
        print(f"  FAILED: {e}")
else:
    # Brute force with formula-guided range
    for cs_guess in [8, 16, 32, 64]:
        T_in = cs_guess * 2 + 13
        x_raw = torch.randn(batch, T_in, 80)
        x_lens = torch.tensor([T_in], dtype=torch.int64)
        try:
            enc_out, enc_lens, new_states = model.encoder.forward(x_raw, x_lens, states)
            print(f"  ✅ chunk_size={cs_guess}, T_in={T_in} WORKS!")
            print(f"     enc_out: {enc_out.shape}, enc_lens: {enc_lens.tolist()}")
            break
        except Exception as e:
            err = str(e).split('\n')[0][:100]
            print(f"  chunk_size={cs_guess}, T_in={T_in} FAIL: {err}")

=== encoder attributes ===
  chunk_size:       32
  left_context_len: 128

=== encoder_embed attributes ===
  convnext.padding: (3, 3)

=== Required input T ===
  chunk_size=32 => T_in needed = 77
  T=75 -> output_frames=31  
  T=76 -> output_frames=31  
  T=77 -> output_frames=32  ✅ MATCH
  T=78 -> output_frames=32  ✅ MATCH
  T=79 -> output_frames=33  
  T=80 -> output_frames=33  

=== encoder.forward with correct T ===
  INPUT:  (B=1, T=77, 80)
  OUTPUT enc_out:  torch.Size([1, 16, 512])
  OUTPUT enc_lens: [16]
  States preserved: True

  ✅ Encoder output dim = 512 (this feeds joiner)


In [30]:
# Cell: Validate decoder and joiner with correct shapes before ONNX export

import torch

model.eval()

batch = 1

# ── Re-run encoder with correct T to get real enc_out ───────────────────────
states = model.encoder.get_init_states(batch_size=batch)
x_raw = torch.randn(batch, 77, 80)
x_lens = torch.tensor([77], dtype=torch.int64)
enc_out, enc_lens, new_states = model.encoder.forward(x_raw, x_lens, states)
print(f"enc_out: {enc_out.shape}")  # (1, 16, 512)

# ── 1. Decoder with context_size=2 ──────────────────────────────────────────
print("\n=== decoder ===")
# context_size=2: feed 2 token IDs (BOS, BOS) as start
y = torch.zeros(batch, 2, dtype=torch.int64)   # (B, context_size=2)

try:
    dec_out = model.decoder.forward(y, need_pad=torch.tensor(True))
    print(f"  INPUT  y:       {y.shape}  (context_size=2)")
    print(f"  OUTPUT dec_out: {dec_out.shape}")  # expect (B, 1, 512)
except Exception as e:
    print(f"  need_pad=True failed: {e}")

try:
    dec_out2 = model.decoder.forward(y, need_pad=torch.tensor(False))
    print(f"  need_pad=False: {dec_out2.shape}")
except Exception as e:
    print(f"  need_pad=False failed: {e}")

# ── 2. simple_lm_proj ────────────────────────────────────────────────────────
print("\n=== simple_lm_proj ===")
lm_out = model.simple_lm_proj.forward(dec_out)
print(f"  INPUT:  {dec_out.shape}")
print(f"  OUTPUT: {lm_out.shape}")   # expect (B, 1, 900)

# ── 3. Joiner — needs raw 512-dim, NOT projected ────────────────────────────
print("\n=== joiner ===")
# enc_out is (B, 16, 512) — take 1 frame for per-step inference
# dec_out is (B,  1, 512)
# joiner: broadcast (B, T, 1, 512) + (B, 1, U, 512) → (B, T, U, vocab)
# For greedy per-step: pass single frames (B,1,512) each

enc_frame = enc_out[:, :1, :]    # (1, 1, 512)
dec_frame = dec_out[:, :1, :]    # (1, 1, 512)

try:
    j_out = model.joiner.forward(enc_frame, dec_frame, project_input=True)
    print(f"  INPUT  enc_frame: {enc_frame.shape}")
    print(f"  INPUT  dec_frame: {dec_frame.shape}")
    print(f"  OUTPUT logits:    {j_out.shape}")   # expect (1,1,1,900)
except Exception as e:
    print(f"  project_input=True failed: {e}")
    try:
        j_out = model.joiner.forward(enc_frame, dec_frame)
        print(f"  (no project_input) OUTPUT: {j_out.shape}")
    except Exception as e2:
        print(f"  no args failed: {e2}")

# ── 4. Full pipeline smoke test ──────────────────────────────────────────────
print("\n=== Full pipeline smoke test ===")
try:
    # Encoder
    enc_out_full, enc_lens_full, _ = model.encoder.forward(x_raw, x_lens, states)
    
    # AM projection (CTC path)
    am_logits = model.simple_am_proj.forward(enc_out_full)
    
    # Decoder
    dec_out_full = model.decoder.forward(y, need_pad=torch.tensor(True))
    lm_logits = model.simple_lm_proj.forward(dec_out_full)
    
    # Joiner (transducer path) — single enc frame × single dec frame
    j_logits = model.joiner.forward(
        enc_out_full[:, :1, :],
        dec_out_full[:, :1, :],
        project_input=True
    )
    
    print(f"  ✅ encoder:        {enc_out_full.shape}")
    print(f"  ✅ am_proj(CTC):   {am_logits.shape}")
    print(f"  ✅ decoder:        {dec_out_full.shape}")
    print(f"  ✅ lm_proj:        {lm_logits.shape}")
    print(f"  ✅ joiner logits:  {j_logits.shape}")
    print(f"\n  🎯 All shapes confirmed — ready for ONNX export!")
except Exception as e:
    print(f"  FAILED: {e}")

enc_out: torch.Size([1, 16, 512])

=== decoder ===
  INPUT  y:       torch.Size([1, 2])  (context_size=2)
  OUTPUT dec_out: torch.Size([1, 2, 512])
  need_pad=False: torch.Size([1, 1, 512])

=== simple_lm_proj ===
  INPUT:  torch.Size([1, 2, 512])
  OUTPUT: torch.Size([1, 2, 900])

=== joiner ===
  INPUT  enc_frame: torch.Size([1, 1, 512])
  INPUT  dec_frame: torch.Size([1, 1, 512])
  OUTPUT logits:    torch.Size([1, 1, 900])

=== Full pipeline smoke test ===
  ✅ encoder:        torch.Size([1, 16, 512])
  ✅ am_proj(CTC):   torch.Size([1, 16, 900])
  ✅ decoder:        torch.Size([1, 2, 512])
  ✅ lm_proj:        torch.Size([1, 2, 900])
  ✅ joiner logits:  torch.Size([1, 1, 900])

  🎯 All shapes confirmed — ready for ONNX export!


In [31]:
# Cell: Correct approach — trace a lambda via torch.jit.trace

import torch
import onnx
import onnxruntime as ort
import numpy as np

model.eval()

batch = 1
context_size = 2
y_dummy = torch.zeros(batch, context_size, dtype=torch.int64)

decoder = model.decoder

# ── Wrap in nn.Module that hardcodes need_pad=False ──────────────────────────
class DecoderInference(torch.nn.Module):
    def __init__(self, decoder):
        super().__init__()
        self.decoder = decoder

    def forward(self, y: torch.Tensor) -> torch.Tensor:
        return self.decoder(y, False)

wrapper = DecoderInference(decoder)
wrapper.eval()

# ── torch.jit.trace bakes False into the graph ───────────────────────────────
with torch.no_grad():
    traced = torch.jit.trace(wrapper, (y_dummy,))

with torch.no_grad():
    out = traced(y_dummy)
    print(f"Traced output: {out.shape}")

# ── Export ────────────────────────────────────────────────────────────────────
DECODER_ONNX = "decoder.onnx"

with torch.no_grad():
    torch.onnx.export(
        traced,
        (y_dummy,),
        DECODER_ONNX,
        input_names=["y"],
        output_names=["decoder_out"],
        dynamic_axes={
            "y":           {0: "batch_size"},
            "decoder_out": {0: "batch_size"},
        },
        opset_version=14,
        do_constant_folding=True,
        dynamo=False,
    )
print(f"✅ Exported: {DECODER_ONNX}")

# ── Validate ──────────────────────────────────────────────────────────────────
onnx_model = onnx.load(DECODER_ONNX)
onnx.checker.check_model(onnx_model)
print(f"✅ ONNX check passed | opset={onnx_model.opset_import[0].version}")

sess = ort.InferenceSession(DECODER_ONNX, providers=["CPUExecutionProvider"])

print(f"\nInputs:")
for inp in sess.get_inputs():
    print(f"  {inp.name}: shape={inp.shape}, dtype={inp.type}")
print(f"Outputs:")
for o in sess.get_outputs():
    print(f"  {o.name}: shape={o.shape}, dtype={o.type}")

# ── Numeric check ─────────────────────────────────────────────────────────────
y_np = np.zeros((batch, context_size), dtype=np.int64)
ort_out = sess.run(["decoder_out"], {"y": y_np})[0]

with torch.no_grad():
    pt_out = decoder.forward(torch.from_numpy(y_np), False).numpy()

max_diff = np.abs(ort_out - pt_out).max()
print(f"\nORT output: {ort_out.shape}")
print(f"Max diff PyTorch vs ORT: {max_diff:.6f}  {'✅ PASS' if max_diff < 1e-4 else '❌ MISMATCH'}")

Traced output: torch.Size([1, 1, 512])
✅ Exported: decoder.onnx
✅ ONNX check passed | opset=14

Inputs:
  y: shape=['batch_size', 2], dtype=tensor(int64)
Outputs:
  decoder_out: shape=['batch_size', 1, 512], dtype=tensor(float)

ORT output: (1, 1, 512)
Max diff PyTorch vs ORT: 0.000000  ✅ PASS


/tmp/ipykernel_58/3372816524.py:40: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1508: UserWarning: no signature found for builtin <built-in method __call__ of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x7ab239711d70>, skipping _decide_input_format
  args = _decide_input_format(model, args)


In [32]:
# Cell: Export joiner to ONNX

import torch
import onnx
import onnxruntime as ort
import numpy as np

model.eval()

batch = 1
joiner = model.joiner

# ── Check joiner forward signature ───────────────────────────────────────────
print("=== joiner code ===")
try:
    print(joiner.forward.code)
except:
    print(joiner.code)

=== joiner code ===
def forward(self,
    encoder_out: Tensor,
    decoder_out: Tensor,
    project_input: bool=True) -> Tensor:
  _0 = torch.eq(torch.dim(encoder_out), torch.dim(decoder_out))
  if _0:
    pass
  else:
    _1 = (torch.size(encoder_out), torch.size(decoder_out))
    _2 = torch.add("AssertionError: ", str(_1))
    ops.prim.RaiseException(_2)
  if project_input:
    encoder_proj = self.encoder_proj
    _3 = (encoder_proj).forward(encoder_out, )
    decoder_proj = self.decoder_proj
    _4 = (decoder_proj).forward(decoder_out, )
    logit = torch.add(_3, _4)
  else:
    logit = torch.add(encoder_out, decoder_out)
  output_linear = self.output_linear
  logit1 = (output_linear).forward(torch.tanh(logit), )
  return logit1



In [33]:
# Cell: After seeing graph — eliminate assert, export clean graph

import torch
import onnx
import onnxruntime as ort
import numpy as np
from torch.onnx import OperatorExportTypes

model.eval()
joiner = model.joiner

batch = 1
enc_dummy = torch.randn(batch, 1, 512)
dec_dummy = torch.randn(batch, 1, 512)

# ── Step 1: Clone graph and inline + optimize to remove dead branches ─────────
graph = joiner.forward.graph

# Freeze project_input=True by running constant propagation after
# manually setting the if-branch
torch._C._jit_pass_inline(graph)
torch._C._jit_pass_constant_propagation(graph)
torch._C._jit_pass_dce(graph)                    # dead code elimination
torch._C._jit_pass_canonicalize(graph)

print("=== Cleaned graph nodes ===")
for i, node in enumerate(g.nodes()):
    print(f"  [{i:03d}] {node.kind():<35} | {str(node)[:80].strip()}")

=== Cleaned graph nodes ===
  [000] prim::Constant                      | %57 : bool = prim::Constant[value=0]()
  [001] prim::Constant                      | %42 : Function = prim::Constant[name="make_pad_mask"]()
  [002] prim::Constant                      | %38 : NoneType = prim::Constant()
  [003] prim::Constant                      | %35 : str = prim::Constant[value="AssertionError: "]() # /nlsasfs/home/nltm-pilo
  [004] prim::Constant                      | %9 : int = prim::Constant[value=-2]() # /nlsasfs/home/nltm-pilot/msdafini/other_
  [005] prim::Constant                      | %23 : int = prim::Constant[value=1]() # /nlsasfs/home/nltm-pilot/msdafini/other_
  [006] prim::Constant                      | %41 : int = prim::Constant[value=0]() # /nlsasfs/home/nltm-pilot/msdafini/other_
  [007] prim::Constant                      | %60 : int = prim::Constant[value=-1]() # /nlsasfs/home/nltm-pilot/msdafini/other
  [008] prim::Constant                      | %79 : int = prim::Consta

In [34]:
# Cell: Cleanest approach — reimplement joiner as plain nn.Module using weight extraction

import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np

model.eval()
joiner = model.joiner

batch = 1
enc_dummy = torch.randn(batch, 1, 512)
dec_dummy = torch.randn(batch, 1, 512)

# ── Extract weights directly from TorchScript submodules ─────────────────────
enc_proj_w  = joiner.encoder_proj.weight.detach()   # (512, 512)
enc_proj_b  = joiner.encoder_proj.bias.detach()     # (512,)
dec_proj_w  = joiner.decoder_proj.weight.detach()   # (512, 512)
dec_proj_b  = joiner.decoder_proj.bias.detach()     # (512,)
out_lin_w   = joiner.output_linear.weight.detach()  # (900, 512)
out_lin_b   = joiner.output_linear.bias.detach()    # (900,)

print(f"encoder_proj : weight={enc_proj_w.shape}, bias={enc_proj_b.shape}")
print(f"decoder_proj : weight={dec_proj_w.shape}, bias={dec_proj_b.shape}")
print(f"output_linear: weight={out_lin_w.shape},  bias={out_lin_b.shape}")

# ── Build clean nn.Module with the same math ──────────────────────────────────
class JoinerONNX(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_proj  = nn.Linear(enc_proj_w.shape[1], enc_proj_w.shape[0])
        self.decoder_proj  = nn.Linear(dec_proj_w.shape[1], dec_proj_w.shape[0])
        self.output_linear = nn.Linear(out_lin_w.shape[1],  out_lin_w.shape[0])

        self.encoder_proj.weight.data  = enc_proj_w
        self.encoder_proj.bias.data    = enc_proj_b
        self.decoder_proj.weight.data  = dec_proj_w
        self.decoder_proj.bias.data    = dec_proj_b
        self.output_linear.weight.data = out_lin_w
        self.output_linear.bias.data   = out_lin_b

    def forward(self, encoder_out: torch.Tensor, decoder_out: torch.Tensor) -> torch.Tensor:
        # project_input=True path only
        logit = self.encoder_proj(encoder_out) + self.decoder_proj(decoder_out)
        return self.output_linear(torch.tanh(logit))

joiner_onnx = JoinerONNX()
joiner_onnx.eval()

# ── Verify numeric match vs original ─────────────────────────────────────────
with torch.no_grad():
    ref  = joiner.forward(enc_dummy, dec_dummy, True)
    ours = joiner_onnx(enc_dummy, dec_dummy)
    diff = (ref - ours).abs().max().item()
    print(f"\nNumeric match vs original joiner: {diff:.8f}  {'✅' if diff < 1e-5 else '❌'}")

# ── Export ────────────────────────────────────────────────────────────────────
JOINER_ONNX = "joiner.onnx"

with torch.no_grad():
    torch.onnx.export(
        joiner_onnx,
        (enc_dummy, dec_dummy),
        JOINER_ONNX,
        input_names=["encoder_out", "decoder_out"],
        output_names=["logits"],
        dynamic_axes={
            "encoder_out": {0: "batch_size"},
            "decoder_out": {0: "batch_size"},
            "logits":      {0: "batch_size"},
        },
        opset_version=14,
        do_constant_folding=True,
        dynamo=False,
    )
print(f"✅ Exported: {JOINER_ONNX}")

# ── Validate ──────────────────────────────────────────────────────────────────
onnx_model = onnx.load(JOINER_ONNX)
onnx.checker.check_model(onnx_model)
print(f"✅ ONNX check passed | opset={onnx_model.opset_import[0].version}")

sess = ort.InferenceSession(JOINER_ONNX, providers=["CPUExecutionProvider"])

print(f"\nInputs:")
for inp in sess.get_inputs():
    print(f"  {inp.name}: shape={inp.shape}, dtype={inp.type}")
print(f"Outputs:")
for o in sess.get_outputs():
    print(f"  {o.name}: shape={o.shape}, dtype={o.type}")

# ── ORT numeric check ─────────────────────────────────────────────────────────
enc_np = enc_dummy.numpy()
dec_np = dec_dummy.numpy()
ort_out = sess.run(["logits"], {"encoder_out": enc_np, "decoder_out": dec_np})[0]

with torch.no_grad():
    pt_out = joiner_onnx(enc_dummy, dec_dummy).numpy()

max_diff = np.abs(ort_out - pt_out).max()
print(f"\nORT output: {ort_out.shape}")
print(f"Max diff PyTorch vs ORT: {max_diff:.6f}  {'✅ PASS' if max_diff < 1e-4 else '❌ MISMATCH'}")

encoder_proj : weight=torch.Size([512, 512]), bias=torch.Size([512])
decoder_proj : weight=torch.Size([512, 512]), bias=torch.Size([512])
output_linear: weight=torch.Size([900, 512]),  bias=torch.Size([900])

Numeric match vs original joiner: 0.00000000  ✅
✅ Exported: joiner.onnx
✅ ONNX check passed | opset=14

Inputs:
  encoder_out: shape=['batch_size', 1, 512], dtype=tensor(float)
  decoder_out: shape=['batch_size', 1, 512], dtype=tensor(float)
Outputs:
  logits: shape=['batch_size', 1, 900], dtype=tensor(float)

ORT output: (1, 1, 900)
Max diff PyTorch vs ORT: 0.000009  ✅ PASS


/tmp/ipykernel_58/3405650459.py:62: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [35]:
# Cell: Export encoder to ONNX with flattened states as individual inputs/outputs

import torch
import onnx
import onnxruntime as ort
import numpy as np

model.eval()

batch = 1
T_in = 77   # chunk_size=32 → needs T=77 raw fbank frames

# ── Get init states and confirm shapes ───────────────────────────────────────
states = model.encoder.get_init_states(batch_size=batch)
print(f"Number of states: {len(states)}")
for i, s in enumerate(states):
    print(f"  state_{i:02d}: {s.shape} {s.dtype}")

Number of states: 98
  state_00: torch.Size([128, 1, 128]) torch.float32
  state_01: torch.Size([1, 1, 128, 144]) torch.float32
  state_02: torch.Size([128, 1, 48]) torch.float32
  state_03: torch.Size([128, 1, 48]) torch.float32
  state_04: torch.Size([1, 192, 15]) torch.float32
  state_05: torch.Size([1, 192, 15]) torch.float32
  state_06: torch.Size([128, 1, 128]) torch.float32
  state_07: torch.Size([1, 1, 128, 144]) torch.float32
  state_08: torch.Size([128, 1, 48]) torch.float32
  state_09: torch.Size([128, 1, 48]) torch.float32
  state_10: torch.Size([1, 192, 15]) torch.float32
  state_11: torch.Size([1, 192, 15]) torch.float32
  state_12: torch.Size([64, 1, 128]) torch.float32
  state_13: torch.Size([1, 1, 64, 192]) torch.float32
  state_14: torch.Size([64, 1, 48]) torch.float32
  state_15: torch.Size([64, 1, 48]) torch.float32
  state_16: torch.Size([1, 256, 15]) torch.float32
  state_17: torch.Size([1, 256, 15]) torch.float32
  state_18: torch.Size([64, 1, 128]) torch.float32

In [36]:
# Cell A: Install sherpa-onnx export tools
!pip install -q sherpa-onnx
!pip install -q kaldifeat 2>/dev/null || echo "kaldifeat optional"

import sherpa_onnx
print(f"sherpa-onnx version: {sherpa_onnx.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 39.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 82.6 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.9/467.9 kB 7.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
kaldifeat optional
sherpa-onnx version: 1.13.0


In [37]:
# Cell B: Use sherpa_onnx to export encoder directly from the .pt
# sherpa-onnx knows how to handle streaming zipformer state management

import torch
import os

PT_PATH = "/kaggle/input/models/nishargnargund/gujarati-asr/pytorch/default/1/SPRING_INX_streaming_k2_Gujarati.pt"
model = torch.jit.load(PT_PATH, map_location="cpu")
model.eval()

# sherpa-onnx export approach: export encoder_embed + encoder together
# as a single "encoder" with fixed state protocol

N_STATES = 98
states = model.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
x_dummy = torch.randn(1, 77, 80)
lens_dummy = torch.tensor([77], dtype=torch.int32)

print(f"States count: {len(states)}")
for i, s in enumerate(states):
    print(f"  state[{i}]: {s.shape} {s.dtype}")

States count: 98
  state[0]: torch.Size([128, 1, 128]) torch.float32
  state[1]: torch.Size([1, 1, 128, 144]) torch.float32
  state[2]: torch.Size([128, 1, 48]) torch.float32
  state[3]: torch.Size([128, 1, 48]) torch.float32
  state[4]: torch.Size([1, 192, 15]) torch.float32
  state[5]: torch.Size([1, 192, 15]) torch.float32
  state[6]: torch.Size([128, 1, 128]) torch.float32
  state[7]: torch.Size([1, 1, 128, 144]) torch.float32
  state[8]: torch.Size([128, 1, 48]) torch.float32
  state[9]: torch.Size([128, 1, 48]) torch.float32
  state[10]: torch.Size([1, 192, 15]) torch.float32
  state[11]: torch.Size([1, 192, 15]) torch.float32
  state[12]: torch.Size([64, 1, 128]) torch.float32
  state[13]: torch.Size([1, 1, 64, 192]) torch.float32
  state[14]: torch.Size([64, 1, 48]) torch.float32
  state[15]: torch.Size([64, 1, 48]) torch.float32
  state[16]: torch.Size([1, 256, 15]) torch.float32
  state[17]: torch.Size([1, 256, 15]) torch.float32
  state[18]: torch.Size([64, 1, 128]) torch.fl

In [38]:
# Cell C (fixed): wrap encoder in nn.Module so tracer can register it as submodule

import torch
import torch.nn as nn

class EncoderTraceWrapper(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder  # registers as submodule ✅

    def forward(self, x, x_lens, *states):
        states_list = list(states)
        enc_out, enc_lens, new_states = self.encoder.forward(x, x_lens, states_list)
        return (enc_out, enc_lens) + tuple(new_states)

N_STATES = 98
states = model.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
x_dummy = torch.randn(1, 77, 80)
lens_dummy = torch.tensor([77], dtype=torch.int32)

wrapper = EncoderTraceWrapper(model.encoder)
wrapper.eval()

# Verify forward works
with torch.no_grad():
    outs = wrapper(x_dummy, lens_dummy, *states)
    print(f"✅ Wrapper forward: enc_out={outs[0].shape}, enc_lens={outs[1]}, n_states={len(outs)-2}")

dummy_args = (x_dummy, lens_dummy) + tuple(states)

with torch.no_grad():
    traced = torch.jit.trace(wrapper, dummy_args, strict=False, check_trace=False)
    outs2 = traced(x_dummy, lens_dummy, *states)
    print(f"✅ Traced: enc_out={outs2[0].shape}")

print("✅ Tracing succeeded — run Cell D next")

✅ Wrapper forward: enc_out=torch.Size([1, 16, 512]), enc_lens=tensor([16], dtype=torch.int32), n_states=98
✅ Traced: enc_out=torch.Size([1, 16, 512])
✅ Tracing succeeded — run Cell D next


In [39]:
# Cell D (final): patch the traced graph then use LEGACY exporter

import torch
import torch.onnx
import onnx
import onnxruntime as ort
import numpy as np
import os

# ── Step 1: get the traced graph and inline it ────────────────────────────────
graph = traced.graph
torch._C._jit_pass_inline(graph)
torch._C._jit_pass_constant_propagation(graph)

# ── Step 2: find and fix all prim::TupleIndex with non-constant index ─────────
# This is the exact op that crashes _jit_pass_lower_all_tuples
def fix_tuple_index(graph):
    changed = 0
    for node in list(graph.nodes()):
        if node.kind() == 'prim::TupleIndex':
            inputs = list(node.inputs())
            idx_node = inputs[1].node()
            if idx_node.kind() != 'prim::Constant':
                # The index is dynamic — but at trace time it was a fixed int
                # Read the concrete value from the trace and bake it in
                try:
                    idx_val = int(inputs[1].toIValue())
                except Exception:
                    continue
                const = graph.insertConstant(idx_val)
                const.node().moveBefore(node)
                inputs[1].replaceAllUsesWith(const)
                changed += 1
    return changed

n = fix_tuple_index(graph)
print(f"Fixed {n} dynamic TupleIndex nodes")

torch._C._jit_pass_constant_propagation(graph)
torch._C._jit_pass_dce(graph)

# ── Step 3: legacy export on the now-patched traced module ────────────────────
N_STATES = 98
input_names  = ["x", "x_lens"] + [f"in_state_{i}" for i in range(N_STATES)]
output_names = ["enc_out", "enc_lens"] + [f"out_state_{i}" for i in range(N_STATES)]

dynamic_axes = {"x": {0: "B"}, "x_lens": {0: "B"}, "enc_out": {0: "B"}, "enc_lens": {0: "B"}}
for i, s in enumerate(states):
    dim = 0 if i >= 96 else 1
    dynamic_axes[f"in_state_{i}"]  = {dim: "B"}
    dynamic_axes[f"out_state_{i}"] = {dim: "B"}

ENCODER_ONNX = "/kaggle/working/encoder.onnx"
dummy_args   = (x_dummy, lens_dummy) + tuple(states)

with torch.no_grad():
    torch.onnx.export(
        traced,
        dummy_args,
        ENCODER_ONNX,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        opset_version=14,
        do_constant_folding=True,
        dynamo=False,
    )

print(f"✅ Exported: {ENCODER_ONNX}  ({os.path.getsize(ENCODER_ONNX)/1e6:.1f} MB)")

onnx_model = onnx.load(ENCODER_ONNX)
onnx.checker.check_model(onnx_model)
print(f"✅ ONNX valid")

sess = ort.InferenceSession(ENCODER_ONNX, providers=["CPUExecutionProvider"])
feed = {"x": x_dummy.numpy(), "x_lens": lens_dummy.numpy()}
for i, s in enumerate(states): feed[f"in_state_{i}"] = s.numpy()
ort_outs = sess.run(None, feed)

with torch.no_grad():
    pt_outs = wrapper(x_dummy, lens_dummy, *states)

max_diff = np.abs(ort_outs[0] - pt_outs[0].numpy()).max()
print(f"✅ enc_out: {ort_outs[0].shape}  max_diff={max_diff:.6f}  {'✅ PASS' if max_diff < 1e-4 else '❌'}")

Fixed 0 dynamic TupleIndex nodes


/tmp/ipykernel_58/353932778.py:57: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Torch IR graph at exception: graph(%x : Float(1, 77, 80, strides=[6160, 80, 1], requires_grad=0, device=cpu),
      %x_lens : Int(1, strides=[1], requires_grad=0, device=cpu),
      %2 : Float(128, 1, 128, strides=[128, 128, 1], requires_grad=0, device=cpu),
      %3 : Float(1, 1, 128, 144, strides=[18432, 18432, 144, 1], requires_grad=0, device=cpu),
      %4 : Float(128, 1, 48, strides=[48, 48, 1], requires_grad=0, device=cpu),
      %5 : Float(128, 1, 48, strides=[48, 48, 1], requires_grad=0, device=cpu),
      %6 : Float(1, 192, 15, strides=[2880, 15, 1], requires_grad=0, device=cpu),
      %7 : Float(1, 192, 15, strides=[2880, 15, 1], requires_grad=0, device=cpu),
      %8 : Float(128, 1, 128, strides=[128, 128, 1], requires_grad=0, device=cpu),
      %9 : Float(1, 1, 128, 144, strides=[18432, 18432, 144, 1], requires_grad=0, device=cpu),
      %10 : Float(128, 1, 48, strides=[48, 48, 1], requires_grad=0, device=cpu),
      %11 : Float(128, 1, 48, strides=[48, 48, 1], requires_gra

[W429 05:25:29.584949636 lower_tuples.cpp:212] Warning: tuple appears in op inputs, but this op does not forward tuples, unsupported kind: aten::str (function flattenInputs)
[W429 05:25:29.585022892 lower_tuples.cpp:212] Warning: tuple appears in op inputs, but this op does not forward tuples, unsupported kind: aten::str (function flattenInputs)
[W429 05:25:29.585029733 lower_tuples.cpp:212] Warning: tuple appears in op inputs, but this op does not forward tuples, unsupported kind: aten::str (function flattenInputs)
[W429 05:25:29.585057361 lower_tuples.cpp:212] Warning: tuple appears in op inputs, but this op does not forward tuples, unsupported kind: aten::str (function flattenInputs)
[W429 05:25:29.585077587 lower_tuples.cpp:212] Warning: tuple appears in op inputs, but this op does not forward tuples, unsupported kind: aten::str (function flattenInputs)
[W429 05:25:29.585086173 lower_tuples.cpp:212] Warning: tuple appears in op inputs, but this op does not forward tuples, unsupport

KeyboardInterrupt: 

d=0, device=cpu),
      %60 : Float(1, 512, 7, strides=[3584, 7, 1], requires_grad=0, device=cpu),
      %61 : Float(1, 512, 7, strides=[3584, 7, 1], requires_grad=0, device=cpu),
      %62 : Float(16, 1, 256, strides=[256, 256, 1], requires_grad=0, device=cpu),
      %63 : Float(1, 1, 16, 384, strides=[6144, 6144, 384, 1], requires_grad=0, device=cpu),
      %64 : Float(16, 1, 96, strides=[96, 96, 1], requires_grad=0, device=cpu),
      %65 : Float(16, 1, 96, strides=[96, 96, 1], requires_grad=0, device=cpu),
      %66 : Float(1, 512, 7, strides=[3584, 7, 1], requires_grad=0, device=cpu),
      %67 : Float(1, 512, 7, strides=[3584, 7, 1], requires_grad=0, device=cpu),
      %68 : Float(32, 1, 128, strides=[128, 128, 1], requires_grad=0, device=cpu),
      %69 : Float(1, 1, 32, 288, strides=[9216, 9216, 288, 1], requires_grad=0, device=cpu),
      %70 : Float(32, 1, 48, strides=[48, 48, 1], requires_grad=0, device=cpu),
      %71 : Float(32, 1, 48, strides=[48, 48, 1], requires_grad=0,

In [40]:
# Cell: Save encoder as TorchScript .pt — done in 10 seconds

import torch
import os

# Save the traced encoder (already works, verified in Cell C)
torch.jit.save(traced, "/kaggle/working/encoder_traced.pt")
print(f"✅ encoder_traced.pt  ({os.path.getsize('/kaggle/working/encoder_traced.pt')/1e6:.1f} MB)")

# Verify decoder and joiner are already exported
for f in ["decoder.onnx", "joiner.onnx"]:
    path = f"/kaggle/working/{f}"
    if os.path.exists(path):
        print(f"✅ {f}  ({os.path.getsize(path)/1e6:.1f} MB)")

# Also export decoder and joiner as .pt for consistency
import torch.onnx

# decoder

decoder_wrapper = torch.jit.trace(
    model.decoder,
    torch.zeros(1, 2, dtype=torch.int64),
    check_trace=False
)
torch.jit.save(decoder_wrapper, "/kaggle/working/decoder_traced.pt")
print(f"✅ decoder_traced.pt")

# joiner  
enc_dummy = torch.randn(1, 1, 1, 512)
dec_dummy = torch.randn(1, 1, 1, 512)
joiner_wrapper = torch.jit.trace(
    model.joiner,
    (enc_dummy, dec_dummy),
    check_trace=False
)
torch.jit.save(joiner_wrapper, "/kaggle/working/joiner_traced.pt")
print(f"✅ joiner_traced.pt")

print("\n✅ ALL 3 .pt files ready for PyTorch Mobile on Android + Raspberry Pi")
print("   encoder_traced.pt  — runs per chunk (77 frames in, 16 frames out + 98 states)")
print("   decoder_traced.pt  — runs per token step")
print("   joiner_traced.pt   — runs per token step")

✅ encoder_traced.pt  (261.3 MB)
✅ decoder.onnx  (1.9 MB)
✅ joiner.onnx  (3.9 MB)
✅ decoder_traced.pt
✅ joiner_traced.pt

✅ ALL 3 .pt files ready for PyTorch Mobile on Android + Raspberry Pi
   encoder_traced.pt  — runs per chunk (77 frames in, 16 frames out + 98 states)
   decoder_traced.pt  — runs per token step
   joiner_traced.pt   — runs per token step


/usr/local/lib/python3.12/dist-packages/torch/jit/_trace.py:1016: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(


In [41]:
import zipfile

zip_path = "/kaggle/working/asr_models.zip"

files = [
    "decoder.onnx",
    "decoder_traced.pt",
    "encoder.onnx",
    "encoder_traced.pt",
    "encoder_wrapper.py",
    "joiner.onnx",
    "joiner_traced.pt"
]

with zipfile.ZipFile(zip_path, 'w') as z:
    for f in files:
        full_path = f"/kaggle/working/{f}"
        if os.path.exists(full_path):
            z.write(full_path, arcname=f)

print("✅ Selective ZIP ready:", zip_path)

✅ Selective ZIP ready: /kaggle/working/asr_models.zip


In [42]:
import os
import torch

def summarize_pt_file(filepath):
    if not os.path.exists(filepath):
        print(f"❌ File not found: {filepath}")
        return

    filename = os.path.basename(filepath)
    print(f"{'='*60}")
    print(f"📄 Model Summary: {filename}")
    print(f"{'='*60}")

    # 1. File Size
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"💾 Disk Size: {size_mb:.2f} MB")

    try:
        # 2. Load the TorchScript model
        model = torch.jit.load(filepath, map_location="cpu")
        model.eval()

        # 3. Parameter Counting
        total_params = sum(p.numel() for p in model.parameters())
        print(f"📊 Total Parameters: {total_params:,}")

        # 4. Extract Methods
        if hasattr(model, '_c') and hasattr(model._c, '_method_names'):
            methods = list(model._c._method_names())
            print(f"🛠️  Exposed Methods: {methods}")
        else:
            print("🛠️  Exposed Methods: Could not automatically detect.")

        # 5. Top-Level Structure
        submodules = list(model.named_children())
        print(f"🧩 Top-level Submodules: {len(submodules)}")
        for name, mod in submodules[:5]:  # Just preview the first 5 to keep it clean
            original_name = getattr(mod, 'original_name', type(mod).__name__)
            print(f"    ├─ {name} ({original_name})")
        if len(submodules) > 5:
            print("    └─ ... (and more)")

    except Exception as e:
        print(f"❌ Error loading {filename}: {e}")
    
    print("\n")

# List of the traced models from your screenshot
pt_files = [
    "encoder_traced.pt",
    "decoder_traced.pt",
    "joiner_traced.pt"
]

print("🔍 Initiating TorchScript Deep-Dive Inspection...\n")
for pt_file in pt_files:
    summarize_pt_file(pt_file)

🔍 Initiating TorchScript Deep-Dive Inspection...

📄 Model Summary: encoder_traced.pt
💾 Disk Size: 249.16 MB
📊 Total Parameters: 64,556,023
🛠️  Exposed Methods: ['forward']
🧩 Top-level Submodules: 1
    ├─ encoder (StreamingEncoderModel)


📄 Model Summary: decoder_traced.pt
💾 Disk Size: 1.79 MB
📊 Total Parameters: 464,896
🛠️  Exposed Methods: ['forward']
🧩 Top-level Submodules: 4
    ├─ embedding (Embedding)
    ├─ balancer (Identity)
    ├─ conv (Conv1d)
    ├─ balancer2 (Identity)


📄 Model Summary: joiner_traced.pt
💾 Disk Size: 3.77 MB
📊 Total Parameters: 987,012
🛠️  Exposed Methods: ['forward']
🧩 Top-level Submodules: 3
    ├─ encoder_proj (Linear)
    ├─ decoder_proj (Linear)
    ├─ output_linear (Linear)




In [43]:
import os
import torch
from torch.utils.mobile_optimizer import optimize_for_mobile

# The files you just inspected
models_to_optimize = {
    "encoder_traced.pt": "encoder.ptl",
    "decoder_traced.pt": "decoder.ptl",
    "joiner_traced.pt": "joiner.ptl"
}

print("🚀 Compiling models for PyTorch Mobile (Edge Deployment)...\n")

for pt_file, ptl_file in models_to_optimize.items():
    
    if os.path.exists(pt_file):
        print(f"Optimizing {pt_file}...")
        try:
            # Load the traced model
            model = torch.jit.load(pt_file, map_location="cpu")

            
            # Run the mobile optimizer (fuses ops, removes JIT overhead)
            optimized_model = optimize_for_mobile(model)
            
            # Save for the Lite Interpreter
            optimized_model._save_for_lite_interpreter(ptl_file)
            print(f"✅ Successfully created: {ptl_file}\n")
        except Exception as e:
            print(f"❌ Error optimizing {pt_file}: {e}\n")
    else:
        print(f"⚠️ Skipping {pt_file} (File not found)\n")

print("🎉 Edge compilation complete! Your .ptl files are ready for Android/Raspberry Pi.")

🚀 Compiling models for PyTorch Mobile (Edge Deployment)...

Optimizing encoder_traced.pt...


/tmp/ipykernel_58/3861085674.py:27: DeprecationWarning: Lite Interpreter is deprecated. Please consider switching to ExecuTorch.             https://docs.pytorch.org/executorch/stable/getting-started.html
  optimized_model._save_for_lite_interpreter(ptl_file)


✅ Successfully created: encoder.ptl

Optimizing decoder_traced.pt...
✅ Successfully created: decoder.ptl

Optimizing joiner_traced.pt...
✅ Successfully created: joiner.ptl

🎉 Edge compilation complete! Your .ptl files are ready for Android/Raspberry Pi.


In [44]:
# Cell: Zip the 3 .ptl files
import zipfile, os

zip_path = "/kaggle/working/asr_models.zip"
files = ["encoder.ptl", "decoder.ptl", "joiner.ptl"]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        full_path = f"/kaggle/working/{f}"
        if os.path.exists(full_path):
            z.write(full_path, arcname=f)
            print(f"  added: {f}  ({os.path.getsize(full_path)/1e6:.1f} MB)")
        else:
            print(f"  ⚠️ missing: {f}")

print(f"\n✅ ZIP ready: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")

# Fast download link via Kaggle
from IPython.display import FileLink
FileLink("/kaggle/working/asr_models.zip")

  added: encoder.ptl  (261.1 MB)
  added: decoder.ptl  (1.9 MB)
  added: joiner.ptl  (4.0 MB)

✅ ZIP ready: /kaggle/working/asr_models.zip  (247.3 MB)


/kaggle/working/asr_models.zip

In [ ]:
from IPython.display import FileLink
display(FileLink("/kaggle/working/asr_models.zip"))

In [ ]:
!pip install -q PyDrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

In [ ]:
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
file = drive.CreateFile({'title': 'asr_models.zip'})
file.SetContentFile('/kaggle/working/asr_models.zip')
file.Upload()

print("Uploaded! File ID:", file['id'])

In [46]:
import torch

# ── Load original .pt for init states ────────────────────────────────────────
PT_PATH = "/kaggle/input/models/nishargnargund/gujarati-asr/pytorch/default/1/SPRING_INX_streaming_k2_Gujarati.pt"
original = torch.jit.load(PT_PATH, map_location="cpu")
original.eval()
print("✅ Original .pt loaded")

# ── Load the 3 .ptl files ─────────────────────────────────────────────────────
encoder = torch.jit.load("/kaggle/working/encoder.ptl", map_location="cpu")
decoder = torch.jit.load("/kaggle/working/decoder.ptl", map_location="cpu")
joiner  = torch.jit.load("/kaggle/working/joiner.ptl",  map_location="cpu")
encoder.eval(); decoder.eval(); joiner.eval()
print("✅ All 3 .ptl loaded")

# ── Init states from original ─────────────────────────────────────────────────
states = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
print(f"✅ Init states: {len(states)}")

# ── Save init states to disk so Android doesn't need the original .pt ─────────
torch.save(states, "/kaggle/working/init_states.pt")
print("✅ init_states.pt saved")

# ── Dummy chunk test ──────────────────────────────────────────────────────────
x      = torch.randn(1, 77, 80)
x_lens = torch.tensor([77], dtype=torch.int32)

with torch.no_grad():
    enc_out_tuple = encoder(x, x_lens, *states)

enc_out    = enc_out_tuple[0]
new_states = list(enc_out_tuple[2:])
print(f"✅ Encoder: {enc_out.shape}")

hyp, results = [0, 0], []
with torch.no_grad():
    for t in range(enc_out.shape[1]):
        y       = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
        dec_out = decoder(y)
        enc_frame = enc_out[:, t:t+1, :].unsqueeze(2)
        dec_frame = dec_out.unsqueeze(1)
        logits    = joiner(enc_frame, dec_frame)
        token     = logits.squeeze().argmax().item()
        if token != 0:
            hyp.append(token); results.append(token)

print(f"✅ Tokens: {results}  (empty on random input = correct)")
print("✅ Pipeline works — ready for Android")

✅ Original .pt loaded
✅ All 3 .ptl loaded
✅ Init states: 98
✅ init_states.pt saved
✅ Encoder: torch.Size([1, 16, 512])


RuntimeError: The following operation failed in the TorchScript interpreter.
Traceback of TorchScript, serialized code (most recent call last):
  File "code/__torch__/decoder/___torch_mangle_99.py", line 9, in forward
    need_pad: bool=True) -> Tensor:
    y0 = torch.to(y, 4)
    _0 = torch.embedding(CONSTANTS.c0, torch.clamp(y0, 0))
         ~~~~~~~~~~~~~~~ <--- HERE
    _1 = torch.unsqueeze(torch.ge(y0, 0), -1)
    embedding_out0 = torch.mul(_0, _1)

Traceback of TorchScript, original code (most recent call last):
  File "/nlsasfs/home/nltm-pilot/msdafini/anaconda3/envs/og_icefall/lib/python3.12/site-packages/torch/nn/functional.py", line 2267, in forward
        # remove once script supports set_grad_enabled
        _no_grad_embedding_renorm_(weight, input, max_norm, norm_type)
    return torch.embedding(weight, input, padding_idx, scale_grad_by_freq, sparse)
           ~~~~~~~~~~~~~~~ <--- HERE
RuntimeError: index out of range in self


In [47]:
# Fix: decoder was traced with (y, need_pad) signature
# need_pad=False is required for inference mode

hyp, results = [0, 0], []
with torch.no_grad():
    for t in range(enc_out.shape[1]):
        y = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
        
        # need_pad=False — critical, matches how decoder was traced
        dec_out = decoder(y, torch.tensor(False))   # [1, 1, 512]
        
        enc_frame = enc_out[:, t:t+1, :].unsqueeze(2)  # [1, 1, 1, 512]
        dec_frame = dec_out.unsqueeze(1)                # [1, 1, 1, 512]
        
        logits = joiner(enc_frame, dec_frame)           # [1, 1, 1, 900]
        token  = logits.squeeze().argmax().item()
        
        if token != 0:
            hyp.append(token); results.append(token)

print(f"✅ Tokens: {results}")
print("✅ Pipeline complete")

✅ Tokens: []
✅ Pipeline complete


In [52]:
token_ids, text = transcribe(
    "/kaggle/input/models/nishargnargund/guj/other/default/1/guj.wav",
    tokens_file="/kaggle/input/datasets/nishargnargund/gujarati-tokens/SPRING_INX_Gujarati_tokens.txt"
)
print(f"Token ids : {token_ids}")
print(f"Transcript: '{text}'")

  Audio: 3.07s → 308 frames → 4 chunks
Token ids : []
Transcript: ''


In [53]:
# Cell: Fix mel extraction to match kaldi/icefall convention

import librosa
import numpy as np
import torch

def audio_to_mel_chunks_fixed(audio_path, sr=16000, n_mels=80, chunk_frames=77):
    wav, _ = librosa.load(audio_path, sr=sr, mono=True)
    
    # Kaldi-compatible mel — matches icefall training exactly
    mel = librosa.feature.melspectrogram(
        y=wav, sr=sr,
        n_fft=400,          # 25ms window at 16kHz
        hop_length=160,     # 10ms shift
        win_length=400,
        n_mels=n_mels,
        fmin=20,
        fmax=8000,
        power=2.0,
        center=False,       # kaldi does not center frames
    )
    
    # Log mel — kaldi uses log(max(mel, floor))
    floor = 1.0
    mel = np.log(np.maximum(mel, floor))
    mel = mel.T  # [T, 80]
    
    # CMVN — per-utterance mean normalization (standard in icefall)
    mel = mel - mel.mean(axis=0, keepdims=True)
    
    T = mel.shape[0]
    print(f"  Audio: {len(wav)/sr:.2f}s → {T} frames")
    
    # Pad to multiple of chunk_frames
    pad = (-T) % chunk_frames
    if pad:
        mel = np.pad(mel, ((0, pad), (0, 0)))
    
    chunks = mel.reshape(-1, chunk_frames, n_mels)
    print(f"  → {len(chunks)} chunks")
    return chunks

# Re-run transcription with fixed mel
def transcribe_fixed(audio_path, tokens_file):
    chunks = audio_to_mel_chunks_fixed(audio_path)
    states = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
    hyp    = [0, 0]
    
    with torch.no_grad():
        for chunk in chunks:
            x      = torch.FloatTensor(chunk).unsqueeze(0)
            x_lens = torch.tensor([77], dtype=torch.int32)
            
            enc_out_tuple = encoder(x, x_lens, *states)
            enc_out = enc_out_tuple[0]
            states  = list(enc_out_tuple[2:])
            
            for t in range(enc_out.shape[1]):
                y         = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
                dec_out   = decoder(y, torch.tensor(False))
                enc_frame = enc_out[:, t:t+1, :].unsqueeze(2)
                dec_frame = dec_out.unsqueeze(1)
                logits    = joiner(enc_frame, dec_frame)
                token     = logits.squeeze().argmax().item()
                if token != 0:
                    hyp.append(token)

    token_ids = hyp[2:]
    
    with open(tokens_file) as f:
        tokens = [line.strip().split()[0] for line in f]
    
    text = ""
    for tid in token_ids:
        if 0 <= tid < len(tokens):
            tok = tokens[tid]
            if tok not in ["<eps>", "<blank>", "<unk>", "<sos/eos>"]:
                text += tok
    
    print(f"Token ids : {token_ids}")
    print(f"Transcript: '{text}'")
    return token_ids, text

TOKENS = "/kaggle/input/datasets/nishargnargund/gujarati-tokens/SPRING_INX_Gujarati_tokens.txt"
WAV    = "/kaggle/input/models/nishargnargund/guj/other/default/1/guj.wav"

token_ids, text = transcribe_fixed(WAV, TOKENS)

  Audio: 3.07s → 305 frames
  → 4 chunks
Token ids : []
Transcript: ''


In [54]:
with open(TOKENS) as f:
    lines = f.readlines()
print(f"Total tokens: {len(lines)}")
print("First 10:", lines[:10])
print("Last 5:",   lines[-5:])

Total tokens: 903
First 10: ['<blk> 0\n', '<sos/eos> 1\n', '<unk> 2\n', '▁છે 3\n', '▁ 4\n', 'ર 5\n', 'ી 6\n', '. 7\n', 'ે 8\n', '▁તો 9\n']
Last 5: ['ૣ 898\n', '’ 899\n', '#0 900\n', '#1 901\n', '#2 902\n']


In [55]:
# Cell: Debug — check raw logit values to see if model is even activating

import torch
import numpy as np

TOKENS = "/kaggle/input/datasets/nishargnargund/gujarati-tokens/SPRING_INX_Gujarati_tokens.txt"
WAV    = "/kaggle/input/models/nishargnargund/guj/other/default/1/guj.wav"

chunks = audio_to_mel_chunks_fixed(WAV)
states = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))

with open(TOKENS) as f:
    tokens = [line.strip().split()[0] for line in f]

with torch.no_grad():
    # Just run first chunk and inspect
    chunk  = chunks[0]
    x      = torch.FloatTensor(chunk).unsqueeze(0)
    x_lens = torch.tensor([77], dtype=torch.int32)

    enc_out_tuple = encoder(x, x_lens, *states)
    enc_out = enc_out_tuple[0]
    states  = list(enc_out_tuple[2:])

    print(f"enc_out stats: min={enc_out.min():.3f} max={enc_out.max():.3f} mean={enc_out.mean():.3f}")

    # Check frame 0
    y         = torch.tensor([[0, 0]], dtype=torch.int64)
    dec_out   = decoder(y, torch.tensor(False))
    enc_frame = enc_out[:, 0:1, :].unsqueeze(2)
    dec_frame = dec_out.unsqueeze(1)
    logits    = joiner(enc_frame, dec_frame).squeeze()  # [900]

    probs = torch.softmax(logits, dim=-1)
    topk  = torch.topk(probs, 10)

    print(f"\nlogits stats: min={logits.min():.3f} max={logits.max():.3f}")
    print(f"\nTop 10 tokens:")
    for prob, idx in zip(topk.values, topk.indices):
        idx = idx.item()
        print(f"  [{idx:4d}] {tokens[idx] if idx < len(tokens) else '?':15s}  prob={prob:.4f}")

    print(f"\nBlank (id=0) prob: {probs[0]:.4f}")

  Audio: 3.07s → 305 frames
  → 4 chunks
enc_out stats: min=-1.325 max=0.609 mean=0.003

logits stats: min=-27.400 max=10.421

Top 10 tokens:
  [   0] <blk>            prob=0.9919
  [   4] ▁                prob=0.0017
  [  12] ▁એ               prob=0.0005
  [  22] ▁અને             prob=0.0005
  [  11] ▁આ               prob=0.0004
  [  61] ▁તે              prob=0.0004
  [  45] ▁હા              prob=0.0004
  [   9] ▁તો              prob=0.0004
  [  51] ▁એક              prob=0.0003
  [  46] ▁ક               prob=0.0002

Blank (id=0) prob: 0.9919


In [56]:
# Cell: Fix — joiner needs project_input=True and correct input shapes

import torch
import numpy as np

TOKENS = "/kaggle/input/datasets/nishargnargund/gujarati-tokens/SPRING_INX_Gujarati_tokens.txt"
WAV    = "/kaggle/input/models/nishargnargund/guj/other/default/1/guj.wav"

with open(TOKENS) as f:
    tokens = [line.strip().split()[0] for line in f]

def transcribe_v3(audio_path):
    chunks = audio_to_mel_chunks_fixed(audio_path)
    states = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
    hyp    = [0, 0]

    with torch.no_grad():
        for chunk in chunks:
            x      = torch.FloatTensor(chunk).unsqueeze(0)
            x_lens = torch.tensor([77], dtype=torch.int32)

            enc_out_tuple = encoder(x, x_lens, *states)
            enc_out = enc_out_tuple[0]   # [1, 16, 512]
            states  = list(enc_out_tuple[2:])

            for t in range(enc_out.shape[1]):
                y         = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
                dec_out   = decoder(y, torch.tensor(False))  # [1, 1, 512]

                # joiner was traced as: joiner(enc, dec, project_input=True)
                # enc: [1, 512], dec: [1, 512] — squeeze to 2D
                enc_frame = enc_out[:, t, :]   # [1, 512]
                dec_frame = dec_out[:, 0, :]   # [1, 512]

                logits = joiner(enc_frame, dec_frame)  # [1, 900]
                token  = logits.squeeze().argmax().item()

                if token != 0:
                    hyp.append(token)

    token_ids = hyp[2:]
    text = ""
    for tid in token_ids:
        if 0 <= tid < len(tokens):
            tok = tokens[tid]
            if tok not in ["<eps>", "<blank>", "<unk>", "<sos/eos>", "#0", "#1", "#2"]:
                text += tok.replace("▁", " ").strip()
    
    print(f"Token ids : {token_ids}")
    print(f"Transcript: '{text.strip()}'")
    return token_ids, text

token_ids, text = transcribe_v3(WAV)

  Audio: 3.07s → 305 frames
  → 4 chunks
Token ids : []
Transcript: ''


In [57]:
# Check joiner's expected input by inspecting its code
print(joiner.forward.code)

def forward(self,
    encoder_out: Tensor,
    decoder_out: Tensor,
    project_input: bool=True) -> Tensor:
  _0 = ops.prepacked.linear_clamp_run(decoder_out, CONSTANTS.c0)
  _1 = ops.prepacked.linear_clamp_run(encoder_out, CONSTANTS.c1)
  _2 = torch.eq(torch.dim(encoder_out), torch.dim(decoder_out))
  if _2:
    pass
  else:
    _3 = (torch.size(encoder_out), torch.size(decoder_out))
    _4 = torch.add("AssertionError: ", str(_3))
    ops.prim.RaiseException(_4)
  if project_input:
    logit = torch.add(_1, _0)
  else:
    logit = torch.add(encoder_out, decoder_out)
  _5 = ops.prepacked.linear_clamp_run(torch.tanh(logit), CONSTANTS.c2)
  return _5



In [58]:
# Cell: Fix joiner call with correct signature

import torch

TOKENS = "/kaggle/input/datasets/nishargnargund/gujarati-tokens/SPRING_INX_Gujarati_tokens.txt"
WAV    = "/kaggle/input/models/nishargnargund/guj/other/default/1/guj.wav"

with open(TOKENS) as f:
    tokens = [line.strip().split()[0] for line in f]

def transcribe_v4(audio_path):
    chunks = audio_to_mel_chunks_fixed(audio_path)
    states = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
    hyp    = [0, 0]

    with torch.no_grad():
        for chunk in chunks:
            x      = torch.FloatTensor(chunk).unsqueeze(0)
            x_lens = torch.tensor([77], dtype=torch.int32)

            enc_out_tuple = encoder(x, x_lens, *states)
            enc_out = enc_out_tuple[0]   # [1, 16, 512]
            states  = list(enc_out_tuple[2:])

            for t in range(enc_out.shape[1]):
                y       = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
                dec_out = decoder(y, torch.tensor(False))  # [1, 1, 512]

                # Both must be exactly [1, 512] — 2D
                enc_frame = enc_out[:, t, :]    # [1, 512]
                dec_frame = dec_out[:, 0, :]    # [1, 512]

                # Pass project_input=True explicitly
                logits = joiner(enc_frame, dec_frame, True)  # [1, 900]
                token  = logits[0].argmax().item()

                if token != 0:
                    hyp.append(token)

    token_ids = hyp[2:]
    text = ""
    for tid in token_ids:
        if 0 <= tid < len(tokens):
            tok = tokens[tid]
            if tok not in ["<eps>", "<blank>", "<unk>", "<sos/eos>", "#0", "#1", "#2"]:
                text += tok.replace("▁", " ")

    print(f"Token ids : {token_ids}")
    print(f"Transcript: '{text.strip()}'")
    return token_ids, text

token_ids, text = transcribe_v4(WAV)

  Audio: 3.07s → 305 frames
  → 4 chunks
Token ids : []
Transcript: ''


In [59]:
# Cell: Compare original model vs ptl pipeline on same input

import torch
import numpy as np

chunks = audio_to_mel_chunks_fixed(WAV)
chunk  = chunks[0]
x      = torch.FloatTensor(chunk).unsqueeze(0)   # [1, 77, 80]
x_lens = torch.tensor([77], dtype=torch.int32)

states = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))

with torch.no_grad():
    # ── Original model encoder ────────────────────────────────────────────────
    orig_enc_out, orig_enc_lens, orig_new_states = original.encoder.forward(x, x_lens, states)
    print(f"Original encoder out: {orig_enc_out.shape}")
    print(f"  min={orig_enc_out.min():.4f} max={orig_enc_out.max():.4f} mean={orig_enc_out.mean():.4f}")

    # ── PTL encoder ───────────────────────────────────────────────────────────
    enc_out_tuple = encoder(x, x_lens, *states)
    ptl_enc_out   = enc_out_tuple[0]
    print(f"\nPTL encoder out: {ptl_enc_out.shape}")
    print(f"  min={ptl_enc_out.min():.4f} max={ptl_enc_out.max():.4f} mean={ptl_enc_out.mean():.4f}")

    # ── Are they the same? ────────────────────────────────────────────────────
    diff = (orig_enc_out - ptl_enc_out).abs().max()
    print(f"\nMax diff original vs ptl encoder: {diff:.6f}")

    # ── Now run full greedy decode on ORIGINAL model directly ─────────────────
    print("\n── Greedy decode with ORIGINAL model ────────────────────────")
    states2 = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
    hyp = [0, 0]

    for chunk in chunks:
        x2     = torch.FloatTensor(chunk).unsqueeze(0)
        x_lens2 = torch.tensor([77], dtype=torch.int32)
        enc_out2, _, states2 = original.encoder.forward(x2, x_lens2, states2)

        for t in range(enc_out2.shape[1]):
            y         = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
            dec_out2  = original.decoder.forward(y, need_pad=False)   # [1, 1, 512]
            enc_frame = enc_out2[:, t, :]   # [1, 512]
            dec_frame = dec_out2[:, 0, :]   # [1, 512]
            logits2   = original.joiner.forward(enc_frame, dec_frame, project_input=True)
            token     = logits2[0].argmax().item()
            if token != 0:
                hyp.append(token)

    token_ids = hyp[2:]
    with open(TOKENS) as f:
        tokens = [line.strip().split()[0] for line in f]
    text = "".join(tokens[t].replace("▁", " ") for t in token_ids
                   if 0 <= t < len(tokens) and tokens[t] not in
                   ["<eps>", "<blank>", "<unk>", "<sos/eos>", "#0", "#1", "#2"])
    print(f"Token ids : {token_ids}")
    print(f"Transcript: '{text.strip()}'")

  Audio: 3.07s → 305 frames
  → 4 chunks
Original encoder out: torch.Size([1, 16, 512])
  min=-1.3248 max=0.6087 mean=0.0030

PTL encoder out: torch.Size([1, 16, 512])
  min=-1.3248 max=0.6087 mean=0.0030

Max diff original vs ptl encoder: 0.000000

── Greedy decode with ORIGINAL model ────────────────────────
Token ids : []
Transcript: ''


In [60]:
# Cell: Use kaldifeat — exact same mel as training

!pip install kaldifeat -q 2>/dev/null || echo "trying alternate..."
!pip install torchaudio -q

import torchaudio
import torchaudio.compliance.kaldi as kaldi
import torch
import numpy as np

TOKENS = "/kaggle/input/datasets/nishargnargund/gujarati-tokens/SPRING_INX_Gujarati_tokens.txt"
WAV    = "/kaggle/input/models/nishargnargund/guj/other/default/1/guj.wav"

with open(TOKENS) as f:
    tokens = [line.strip().split()[0] for line in f]

def audio_to_mel_kaldi(audio_path, chunk_frames=77):
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)

    # Exact kaldi fbank — matches icefall/SPRING-INX training
    mel = kaldi.fbank(
        waveform,
        num_mel_bins=80,
        frame_length=25.0,    # ms
        frame_shift=10.0,     # ms
        high_freq=8000,
        low_freq=20,
        sample_frequency=16000,
        use_energy=False,
    )  # [T, 80]

    print(f"  Audio: {waveform.shape[1]/16000:.2f}s → {mel.shape[0]} frames")

    # Per-utterance CMVN
    mel = mel - mel.mean(0)

    T   = mel.shape[0]
    pad = (-T) % chunk_frames
    if pad > 0:
        mel = torch.cat([mel, torch.zeros(pad, 80)], dim=0)

    chunks = mel.reshape(-1, chunk_frames, 80)
    print(f"  → {len(chunks)} chunks")
    return chunks

def transcribe_kaldi(audio_path):
    chunks = audio_to_mel_kaldi(audio_path)
    states = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
    hyp    = [0, 0]

    with torch.no_grad():
        for chunk in chunks:
            x      = chunk.unsqueeze(0)                       # [1, 77, 80]
            x_lens = torch.tensor([77], dtype=torch.int32)

            enc_out_tuple = encoder(x, x_lens, *states)
            enc_out = enc_out_tuple[0]
            states  = list(enc_out_tuple[2:])

            for t in range(enc_out.shape[1]):
                y         = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
                dec_out   = decoder(y, torch.tensor(False))
                enc_frame = enc_out[:, t, :]
                dec_frame = dec_out[:, 0, :]
                logits    = joiner(enc_frame, dec_frame, True)
                token     = logits[0].argmax().item()
                if token != 0:
                    hyp.append(token)

    token_ids = hyp[2:]
    text = "".join(
        tokens[t].replace("▁", " ")
        for t in token_ids
        if 0 <= t < len(tokens) and
        tokens[t] not in ["<eps>","<blank>","<unk>","<sos/eos>","#0","#1","#2"]
    )
    print(f"Token ids : {token_ids}")
    print(f"Transcript: '{text.strip()}'")
    return token_ids, text

token_ids, text = transcribe_kaldi(WAV)

  Preparing metadata (setup.py) ... done
trying alternate...
  Audio: 3.07s → 305 frames
  → 4 chunks
Token ids : []
Transcript: ''


In [61]:
# Cell: Test with exact features from original model's feature pipeline

import torchaudio
import torchaudio.compliance.kaldi as kaldi
import torch

WAV    = "/kaggle/input/models/nishargnargund/guj/other/default/1/guj.wav"
TOKENS = "/kaggle/input/datasets/nishargnargund/gujarati-tokens/SPRING_INX_Gujarati_tokens.txt"

with open(TOKENS) as f:
    tokens = [line.strip().split()[0] for line in f]

waveform, sr = torchaudio.load(WAV)
if sr != 16000:
    waveform = torchaudio.functional.resample(waveform, sr, 16000)
if waveform.shape[0] > 1:
    waveform = waveform.mean(0, keepdim=True)

# No CMVN this time
mel = kaldi.fbank(
    waveform,
    num_mel_bins=80,
    frame_length=25.0,
    frame_shift=10.0,
    high_freq=8000,
    low_freq=20,
    sample_frequency=16000,
    use_energy=False,
)  # [T, 80]

print(f"Mel shape: {mel.shape}")
print(f"Mel stats: min={mel.min():.3f} max={mel.max():.3f} mean={mel.mean():.3f}")

# ── Try full utterance at once (no chunking) with original model ──────────────
with torch.no_grad():
    # Pass entire mel as one big sequence to original model's encoder_embed
    x      = mel.unsqueeze(0)                                    # [1, T, 80]
    x_lens = torch.tensor([mel.shape[0]], dtype=torch.int32)

    # Use encoder_embed to check feature processing
    states    = original.encoder.get_init_states(batch_size=1)
    embed_state = states[-2]

    x_embed, x_embed_lens, _ = original.encoder_embed.streaming_forward(
        x, x_lens, embed_state
    )
    print(f"\nAfter encoder_embed: {x_embed.shape}")
    print(f"embed stats: min={x_embed.min():.3f} max={x_embed.max():.3f} mean={x_embed.mean():.3f}")

# ── Now try chunked with NO normalization ────────────────────────────────────
print("\n── Chunked decode, NO CMVN ──────────────────────────────────")
CHUNK = 77
T     = mel.shape[0]
pad   = (-T) % CHUNK
if pad:
    mel_pad = torch.cat([mel, torch.zeros(pad, 80)], dim=0)
else:
    mel_pad = mel

chunks = mel_pad.reshape(-1, CHUNK, 80)
states = original.encoder.get_init_states(batch_size=1)
hyp    = [0, 0]

with torch.no_grad():
    for chunk in chunks:
        x      = chunk.unsqueeze(0)
        x_lens = torch.tensor([77], dtype=torch.int32)

        # Use ORIGINAL model submodules directly
        enc_out2, _, states = original.encoder.forward(x, x_lens, states)

        for t in range(enc_out2.shape[1]):
            y         = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
            dec_out2  = original.decoder.forward(y, need_pad=False)
            enc_frame = enc_out2[:, t, :]
            dec_frame = dec_out2[:, 0, :]
            logits    = original.joiner.forward(enc_frame, dec_frame, project_input=True)
            token     = logits[0].argmax().item()
            if token != 0:
                hyp.append(token)

token_ids = hyp[2:]
text = "".join(
    tokens[t].replace("▁", " ")
    for t in token_ids
    if 0 <= t < len(tokens) and
    tokens[t] not in ["<eps>","<blank>","<unk>","<sos/eos>","#0","#1","#2"]
)
print(f"Token ids: {token_ids}")
print(f"Transcript: '{text.strip()}'")

Mel shape: torch.Size([305, 80])
Mel stats: min=-15.942 max=6.075 mean=-9.784

After encoder_embed: torch.Size([1, 146, 192])
embed stats: min=-1.471 max=1.526 mean=-0.008

── Chunked decode, NO CMVN ──────────────────────────────────
Token ids: [390, 9]
Transcript: 'કેમ તો'


In [62]:
# Cell: Final working transcription pipeline

import torchaudio
import torchaudio.compliance.kaldi as kaldi
import torch

TOKENS = "/kaggle/input/datasets/nishargnargund/gujarati-tokens/SPRING_INX_Gujarati_tokens.txt"

with open(TOKENS) as f:
    tokens = [line.strip().split()[0] for line in f]

def transcribe_final(audio_path):
    # Load audio
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)

    # Kaldi fbank — NO CMVN (this was the bug all along)
    mel = kaldi.fbank(
        waveform,
        num_mel_bins=80,
        frame_length=25.0,
        frame_shift=10.0,
        high_freq=8000,
        low_freq=20,
        sample_frequency=16000,
        use_energy=False,
    )  # [T, 80]

    # Chunk into 77-frame pieces
    CHUNK = 77
    T     = mel.shape[0]
    pad   = (-T) % CHUNK
    if pad:
        mel = torch.cat([mel, torch.zeros(pad, 80)], dim=0)
    chunks = mel.reshape(-1, CHUNK, 80)

    print(f"  Audio: {waveform.shape[1]/16000:.2f}s → {T} frames → {len(chunks)} chunks")

    # Init states from original model
    states = original.encoder.get_init_states(batch_size=1)
    hyp    = [0, 0]

    with torch.no_grad():
        for chunk in chunks:
            x      = chunk.unsqueeze(0)
            x_lens = torch.tensor([77], dtype=torch.int32)

            enc_out_tuple = encoder(x, x_lens, *states)
            enc_out = enc_out_tuple[0]
            states  = list(enc_out_tuple[2:])

            for t in range(enc_out.shape[1]):
                y         = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
                dec_out   = decoder(y, torch.tensor(False))
                enc_frame = enc_out[:, t, :]
                dec_frame = dec_out[:, 0, :]
                logits    = joiner(enc_frame, dec_frame, True)
                token     = logits[0].argmax().item()
                if token != 0:
                    hyp.append(token)

    token_ids = hyp[2:]
    text = "".join(
        tokens[t].replace("▁", " ")
        for t in token_ids
        if 0 <= t < len(tokens) and
        tokens[t] not in ["<eps>", "<blank>", "<unk>", "<sos/eos>", "#0", "#1", "#2"]
    )
    return text.strip()

# Test
WAV  = "/kaggle/input/models/nishargnargund/guj/other/default/1/guj.wav"
text = transcribe_final(WAV)
print(f"✅ Transcript: '{text}'")

  Audio: 3.07s → 305 frames → 4 chunks
✅ Transcript: 'કેમ તો'


In [63]:
# Cell: Save init states as a single tensor bundle for Android

import torch

# Get init states
states = original.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))

# Save as a dict of named tensors — easy to load on Android
state_dict = {f"state_{i}": s for i, s in enumerate(states)}
torch.save(state_dict, "/kaggle/working/init_states.pt")

print(f"✅ Saved {len(states)} init states")
for i, s in enumerate(states):
    print(f"  state_{i}: {s.shape} {s.dtype}")

✅ Saved 98 init states
  state_0: torch.Size([128, 1, 128]) torch.float32
  state_1: torch.Size([1, 1, 128, 144]) torch.float32
  state_2: torch.Size([128, 1, 48]) torch.float32
  state_3: torch.Size([128, 1, 48]) torch.float32
  state_4: torch.Size([1, 192, 15]) torch.float32
  state_5: torch.Size([1, 192, 15]) torch.float32
  state_6: torch.Size([128, 1, 128]) torch.float32
  state_7: torch.Size([1, 1, 128, 144]) torch.float32
  state_8: torch.Size([128, 1, 48]) torch.float32
  state_9: torch.Size([128, 1, 48]) torch.float32
  state_10: torch.Size([1, 192, 15]) torch.float32
  state_11: torch.Size([1, 192, 15]) torch.float32
  state_12: torch.Size([64, 1, 128]) torch.float32
  state_13: torch.Size([1, 1, 64, 192]) torch.float32
  state_14: torch.Size([64, 1, 48]) torch.float32
  state_15: torch.Size([64, 1, 48]) torch.float32
  state_16: torch.Size([1, 256, 15]) torch.float32
  state_17: torch.Size([1, 256, 15]) torch.float32
  state_18: torch.Size([64, 1, 128]) torch.float32
  state